In [1]:
# Task 6 — Question Answering with Transformers
# Step 1: Environment Audit
# Run this cell first and send me the complete output.

import sys
import platform
import subprocess
import importlib.util

print("=" * 70)
print("QUESTION ANSWERING WITH TRANSFORMERS — ENVIRONMENT AUDIT")
print("=" * 70)

# Python
print("\n[1] Python")
print(f"Version       : {sys.version}")
print(f"Executable    : {sys.executable}")
print(f"Platform      : {platform.platform()}")

# PyTorch
print("\n[2] PyTorch")
try:
    import torch

    print(f"Version       : {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"CUDA version  : {torch.version.cuda}")
        print(f"GPU count     : {torch.cuda.device_count()}")

        for i in range(torch.cuda.device_count()):
            gpu = torch.cuda.get_device_properties(i)
            print(f"GPU {i}         : {gpu.name}")
            print(f"GPU memory    : {gpu.total_memory / (1024**3):.2f} GB")
    else:
        print("GPU           : Not available")

except Exception as e:
    print(f"PyTorch error : {type(e).__name__}: {e}")

# Required libraries
print("\n[3] Required Libraries")

required_packages = [
    "transformers",
    "datasets",
    "evaluate",
    "tokenizers",
    "pandas",
    "numpy",
    "accelerate",
    "streamlit",
]

for package in required_packages:
    spec = importlib.util.find_spec(package)

    if spec is None:
        print(f"{package:<15}: NOT INSTALLED")
        continue

    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "version unavailable")
        print(f"{package:<15}: {version}")
    except Exception as e:
        print(f"{package:<15}: INSTALLED — import error: {type(e).__name__}: {e}")

# Transformers backend details
print("\n[4] Transformers Backend")

try:
    import transformers

    print(f"Transformers version: {transformers.__version__}")
    print(f"PyTorch available   : {transformers.is_torch_available()}")
    print(f"TensorFlow available: {transformers.is_tf_available()}")

except Exception as e:
    print(f"Transformers inspection error: {type(e).__name__}: {e}")

# GPU diagnostic
print("\n[5] GPU Diagnostic")

if "torch" in sys.modules and torch.cuda.is_available():
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        text=True
    ))
else:
    print("nvidia-smi check skipped because CUDA is unavailable.")

print("=" * 70)
print("AUDIT COMPLETE")
print("=" * 70)

QUESTION ANSWERING WITH TRANSFORMERS — ENVIRONMENT AUDIT

[1] Python
Version       : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Executable    : /usr/bin/python3
Platform      : Linux-6.6.122+-x86_64-with-glibc2.39

[2] PyTorch
Version       : 2.11.0+cu128
CUDA available: True
CUDA version  : 12.8
GPU count     : 1
GPU 0         : Tesla T4
GPU memory    : 14.56 GB

[3] Required Libraries
transformers   : 5.16.1
datasets       : 4.8.5
evaluate       : NOT INSTALLED
tokenizers     : 0.23.1
pandas         : 2.2.3
numpy          : 2.1.3
accelerate     : 1.14.0
streamlit      : NOT INSTALLED

[4] Transformers Backend
Transformers version: 5.16.1
PyTorch available   : True
Transformers inspection error: AttributeError: module transformers has no attribute is_tf_available

[5] GPU Diagnostic
Tesla T4, 15360 MiB, 580.82.07

AUDIT COMPLETE


In [2]:
# Task 6 — Question Answering with Transformers
# Step 2: Install only the missing project dependencies

import sys
import subprocess
import importlib.util

required_missing = {
    "evaluate": "evaluate",
    "streamlit": "streamlit",
}

missing = [
    package
    for package, import_name in required_missing.items()
    if importlib.util.find_spec(import_name) is None
]

if missing:
    print("Missing packages detected:")
    for package in missing:
        print(f"  - {package}")

    print("\nInstalling only missing packages...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing
    ])

    print("\nInstallation completed.")
else:
    print("All required packages are already installed.")

print("\nVerifying installations...")

for package, import_name in required_missing.items():
    try:
        module = __import__(import_name)
        version = getattr(module, "__version__", "version unavailable")
        print(f"{package:<12}: {version}")
    except Exception as e:
        print(f"{package:<12}: ERROR — {type(e).__name__}: {e}")

Missing packages detected:
  - evaluate
  - streamlit

Installing only missing packages...

Installation completed.

Verifying installations...
evaluate    : 0.4.6
streamlit   : 1.64.0


In [3]:
# Task 6 — Question Answering with Transformers
# Step 3: Load and verify SQuAD v1.1

from datasets import load_dataset

print("=" * 70)
print("LOADING SQUAD V1.1")
print("=" * 70)

# Load the official SQuAD v1.1 dataset from Hugging Face
squad = load_dataset("rajpurkar/squad")

print("\nDataset loaded successfully.")

# Display dataset structure
print("\n[1] Dataset structure")
print(squad)

# Display split information
print("\n[2] Dataset splits")
for split_name, split_data in squad.items():
    print(f"{split_name:<12}: {len(split_data):,} examples")

# Display column names
print("\n[3] Columns")
print(squad["train"].column_names)

# Inspect one training example
example = squad["train"][0]

print("\n[4] First training example")
print("-" * 70)

print(f"ID       : {example['id']}")
print(f"Title    : {example['title']}")
print(f"Question : {example['question']}")

print("\nContext:")
print(example["context"])

print("\nAnswers:")
print(example["answers"])

# Verify expected SQuAD fields
expected_columns = {
    "id",
    "title",
    "context",
    "question",
    "answers",
}

actual_columns = set(squad["train"].column_names)

print("\n[5] Schema verification")

if expected_columns.issubset(actual_columns):
    print("PASS — Expected SQuAD fields are present.")
else:
    missing = expected_columns - actual_columns
    print(f"FAIL — Missing fields: {missing}")

# Verify answer structure
answers = example["answers"]

print("\n[6] Answer structure verification")

print(f"Answer text  : {answers['text']}")
print(f"Answer start : {answers['answer_start']}")

if len(answers["text"]) == len(answers["answer_start"]):
    print("PASS — Answer text and answer_start are aligned.")
else:
    print("FAIL — Answer text and answer_start are not aligned.")

print("\n" + "=" * 70)
print("SQUAD VERIFICATION COMPLETE")
print("=" * 70)

LOADING SQUAD V1.1


README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 14.5MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 1.82MB            

plain_text/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]


Dataset loaded successfully.

[1] Dataset structure
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

[2] Dataset splits
train       : 87,599 examples
validation  : 10,570 examples

[3] Columns
['id', 'title', 'context', 'question', 'answers']

[4] First training example
----------------------------------------------------------------------
ID       : 5733be284776f41900661182
Title    : University_of_Notre_Dame
Question : To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

Context:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next 

In [4]:
# Task 6 — Question Answering with Transformers
# Step 4: Exploratory Data Analysis (EDA)

import pandas as pd
import numpy as np

print("=" * 70)
print("SQUAD V1.1 — EXPLORATORY DATA ANALYSIS")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Basic dataset statistics
# ---------------------------------------------------------------------

train_size = len(squad["train"])
validation_size = len(squad["validation"])

print("\n[1] Dataset size")
print(f"Training examples   : {train_size:,}")
print(f"Validation examples : {validation_size:,}")

# ---------------------------------------------------------------------
# 2. Inspect answer structure across the dataset
# ---------------------------------------------------------------------

print("\n[2] Answer structure")

train_answers = squad["train"]["answers"]
validation_answers = squad["validation"]["answers"]

train_answer_counts = [len(a["text"]) for a in train_answers]
validation_answer_counts = [len(a["text"]) for a in validation_answers]

print(f"Train examples with at least one answer : {sum(c > 0 for c in train_answer_counts):,}")
print(f"Validation examples with at least one answer : {sum(c > 0 for c in validation_answer_counts):,}")

print(f"Maximum answers for one train example : {max(train_answer_counts)}")
print(f"Maximum answers for one validation example : {max(validation_answer_counts)}")

# ---------------------------------------------------------------------
# 3. Context, question, and answer lengths
# ---------------------------------------------------------------------

print("\n[3] Text length statistics")

train_context_lengths = np.array([
    len(context.split())
    for context in squad["train"]["context"]
])

train_question_lengths = np.array([
    len(question.split())
    for question in squad["train"]["question"]
])

train_answer_lengths = np.array([
    len(answer["text"][0].split()) if answer["text"] else 0
    for answer in train_answers
])

print("\nTraining context length (words)")
print(f"Mean   : {train_context_lengths.mean():.2f}")
print(f"Median : {np.median(train_context_lengths):.2f}")
print(f"Max    : {train_context_lengths.max():,}")

print("\nTraining question length (words)")
print(f"Mean   : {train_question_lengths.mean():.2f}")
print(f"Median : {np.median(train_question_lengths):.2f}")
print(f"Max    : {train_question_lengths.max():,}")

print("\nTraining answer length (words)")
print(f"Mean   : {train_answer_lengths.mean():.2f}")
print(f"Median : {np.median(train_answer_lengths):.2f}")
print(f"Max    : {train_answer_lengths.max():,}")

# ---------------------------------------------------------------------
# 4. Unique contexts
# ---------------------------------------------------------------------

print("\n[4] Context statistics")

unique_train_contexts = len(set(squad["train"]["context"]))

print(f"Unique training contexts : {unique_train_contexts:,}")
print(f"Training examples        : {train_size:,}")

questions_per_context = train_size / unique_train_contexts

print(f"Average questions per unique context : {questions_per_context:.2f}")

# ---------------------------------------------------------------------
# 5. Answer span validation
# ---------------------------------------------------------------------

print("\n[5] Answer span validation")

invalid_spans = 0
checked_examples = 0

for example in squad["train"]:
    context = example["context"]
    answers = example["answers"]

    for answer_text, answer_start in zip(
        answers["text"],
        answers["answer_start"]
    ):
        checked_examples += 1

        extracted_text = context[
            answer_start: answer_start + len(answer_text)
        ]

        if extracted_text != answer_text:
            invalid_spans += 1

print(f"Answer spans checked : {checked_examples:,}")
print(f"Invalid spans        : {invalid_spans:,}")

if invalid_spans == 0:
    print("PASS — All checked answer spans match their context.")
else:
    print("WARNING — Some answer spans do not exactly match their context.")

# ---------------------------------------------------------------------
# 6. Create a compact EDA table
# ---------------------------------------------------------------------

eda_summary = pd.DataFrame({
    "Metric": [
        "Training examples",
        "Validation examples",
        "Unique training contexts",
        "Average questions per context",
        "Mean context length",
        "Median context length",
        "Maximum context length",
        "Mean question length",
        "Median question length",
        "Maximum question length",
        "Mean answer length",
        "Median answer length",
        "Maximum answer length",
        "Invalid training answer spans",
    ],
    "Value": [
        train_size,
        validation_size,
        unique_train_contexts,
        round(questions_per_context, 2),
        round(train_context_lengths.mean(), 2),
        round(np.median(train_context_lengths), 2),
        int(train_context_lengths.max()),
        round(train_question_lengths.mean(), 2),
        round(np.median(train_question_lengths), 2),
        int(train_question_lengths.max()),
        round(train_answer_lengths.mean(), 2),
        round(np.median(train_answer_lengths), 2),
        int(train_answer_lengths.max()),
        invalid_spans,
    ]
})

print("\n[6] EDA summary")
display(eda_summary)

print("\n" + "=" * 70)
print("EDA COMPLETE")
print("=" * 70)

SQUAD V1.1 — EXPLORATORY DATA ANALYSIS

[1] Dataset size
Training examples   : 87,599
Validation examples : 10,570

[2] Answer structure
Train examples with at least one answer : 87,599
Validation examples with at least one answer : 10,570
Maximum answers for one train example : 1
Maximum answers for one validation example : 6

[3] Text length statistics

Training context length (words)
Mean   : 119.76
Median : 110.00
Max    : 653

Training question length (words)
Mean   : 10.06
Median : 10.00
Max    : 40

Training answer length (words)
Mean   : 3.16
Median : 2.00
Max    : 43

[4] Context statistics
Unique training contexts : 18,891
Training examples        : 87,599
Average questions per unique context : 4.64

[5] Answer span validation
Answer spans checked : 87,599
Invalid spans        : 0
PASS — All checked answer spans match their context.

[6] EDA summary


,Metric,Value
0,Training examples,87599.00
1,Validation examples,10570.00
2,Unique training contexts,18891.00
3,Average questions per context,4.64
4,Mean context length,119.76
5,Median context length,110.00
6,Maximum context length,653.00
7,Mean question length,10.06
8,Median question length,10.00
9,Maximum question length,40.00



EDA COMPLETE


In [5]:
# Task 6 — Question Answering with Transformers
# Step 5: Load the DistilBERT tokenizer and inspect offset mappings

from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

print("=" * 70)
print("DISTILBERT TOKENIZER INSPECTION")
print("=" * 70)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\n[1] Tokenizer information")
print(f"Model name       : {MODEL_NAME}")
print(f"Tokenizer class   : {tokenizer.__class__.__name__}")
print(f"Vocabulary size   : {tokenizer.vocab_size}")
print(f"Fast tokenizer    : {tokenizer.is_fast}")

# Select a real SQuAD example
example = squad["train"][0]

question = example["question"]
context = example["context"]
answer_text = example["answers"]["text"][0]
answer_start = example["answers"]["answer_start"][0]
answer_end = answer_start + len(answer_text)

print("\n[2] Original example")
print(f"Question     : {question}")
print(f"Answer       : {answer_text}")
print(f"Answer start : {answer_start}")
print(f"Answer end   : {answer_end}")

# Tokenize question + context together
tokenized = tokenizer(
    question,
    context,
    truncation="only_second",
    max_length=384,
    padding=False,
    return_offsets_mapping=True,
    return_special_tokens_mask=True,
)

print("\n[3] Tokenization information")
print(f"Number of tokens : {len(tokenized['input_ids'])}")
print(f"Input IDs        : {tokenized['input_ids'][:20]} ...")

# Convert IDs back to readable tokens
tokens = tokenizer.convert_ids_to_tokens(tokenized["input_ids"])

print("\n[4] First 40 tokens")
for i, token in enumerate(tokens[:40]):
    print(f"{i:>4}: {token}")

# Inspect token type / sequence information
sequence_ids = tokenized.sequence_ids()

print("\n[5] Sequence mapping")
print("0 = question")
print("1 = context")
print("None = special token")

for i in range(min(30, len(tokens))):
    print(
        f"{i:>4}: "
        f"token={tokens[i]!r:<18} "
        f"sequence_id={sequence_ids[i]!s:<5} "
        f"offset={tokenized['offset_mapping'][i]}"
    )

# ---------------------------------------------------------------------
# Locate the answer using character offsets
# ---------------------------------------------------------------------

print("\n[6] Answer span inspection")

answer_token_positions = []

for idx, (offset, sequence_id) in enumerate(
    zip(tokenized["offset_mapping"], sequence_ids)
):
    start, end = offset

    # Only context tokens have sequence_id == 1.
    # Check whether the token overlaps the character-level answer span.
    if sequence_id == 1 and start < answer_end and end > answer_start:
        answer_token_positions.append(idx)

if answer_token_positions:
    token_start = answer_token_positions[0]
    token_end = answer_token_positions[-1]

    print(f"Character answer span : {answer_start}:{answer_end}")
    print(f"Token answer span     : {token_start}:{token_end}")

    print("\nAnswer tokens:")
    for idx in range(token_start, token_end + 1):
        print(
            f"{idx:>4}: "
            f"{tokens[idx]!r:<20} "
            f"offset={tokenized['offset_mapping'][idx]}"
        )

    reconstructed_answer = tokenizer.decode(
        tokenized["input_ids"][token_start:token_end + 1],
        skip_special_tokens=True,
    )

    print(f"\nOriginal answer       : {answer_text!r}")
    print(f"Reconstructed tokens  : {reconstructed_answer!r}")

else:
    print("Answer span was not found inside the tokenized context.")

print("\n" + "=" * 70)
print("TOKENIZER INSPECTION COMPLETE")
print("=" * 70)

DISTILBERT TOKENIZER INSPECTION


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


[1] Tokenizer information
Model name       : distilbert-base-uncased
Tokenizer class   : BertTokenizer
Vocabulary size   : 30522
Fast tokenizer    : True

[2] Original example
Question     : To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer       : Saint Bernadette Soubirous
Answer start : 515
Answer end   : 541

[3] Tokenization information
Number of tokens : 176
Input IDs        : [101, 2000, 3183, 2106, 1996, 6261, 2984, 9382, 3711, 1999, 8517, 1999, 10223, 26371, 2605, 1029, 102, 6549, 2135, 1010] ...

[4] First 40 tokens
   0: [CLS]
   1: to
   2: whom
   3: did
   4: the
   5: virgin
   6: mary
   7: allegedly
   8: appear
   9: in
  10: 1858
  11: in
  12: lou
  13: ##rdes
  14: france
  15: ?
  16: [SEP]
  17: architectural
  18: ##ly
  19: ,
  20: the
  21: school
  22: has
  23: a
  24: catholic
  25: character
  26: .
  27: atop
  28: the
  29: main
  30: building
  31: '
  32: s
  33: gold
  34: dome
  35: is
  36: a
  37: golden
  38: statue
 

In [6]:
# Task 6 — Question Answering with Transformers
# Step 6: Inspect long-context handling with overflow and stride

import numpy as np

print("=" * 70)
print("LONG-CONTEXT HANDLING — OVERFLOW + STRIDE")
print("=" * 70)

MAX_LENGTH = 384
DOC_STRIDE = 128

# Find the longest training context
longest_example = max(
    squad["train"],
    key=lambda example: len(example["context"].split())
)

long_context = longest_example["context"]
long_question = longest_example["question"]

print("\n[1] Longest training context")
print(f"Question           : {long_question}")
print(f"Context words      : {len(long_context.split()):,}")
print(f"Context characters : {len(long_context):,}")

# Tokenize with overflow
tokenized_long = tokenizer(
    long_question,
    long_context,
    truncation="only_second",
    max_length=MAX_LENGTH,
    stride=DOC_STRIDE,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

num_windows = len(tokenized_long["input_ids"])

print("\n[2] Overflow information")
print(f"Max sequence length : {MAX_LENGTH}")
print(f"Document stride     : {DOC_STRIDE}")
print(f"Number of windows   : {num_windows}")

# Show window sizes
window_lengths = [
    len(input_ids)
    for input_ids in tokenized_long["input_ids"]
]

print(f"Window lengths      : {window_lengths}")

# Map each feature/window back to its original example
overflow_mapping = tokenized_long["overflow_to_sample_mapping"]

print("\n[3] Overflow mapping")
print(f"Mapping length      : {len(overflow_mapping)}")
print(f"Unique source IDs   : {sorted(set(overflow_mapping))}")

# Inspect the context coverage of each window
print("\n[4] Context coverage per window")

for window_idx in range(num_windows):
    offsets = tokenized_long["offset_mapping"][window_idx]
    sequence_ids = tokenized_long.sequence_ids(window_idx)

    context_offsets = [
        offset
        for offset, sequence_id in zip(offsets, sequence_ids)
        if sequence_id == 1 and offset[1] > offset[0]
    ]

    if context_offsets:
        context_start = min(offset[0] for offset in context_offsets)
        context_end = max(offset[1] for offset in context_offsets)

        print(
            f"Window {window_idx:>2}: "
            f"character range {context_start:,}:{context_end:,} "
            f"({context_end - context_start:,} characters)"
        )

# ---------------------------------------------------------------------
# Verify the actual answer span across overflow windows
# ---------------------------------------------------------------------

answer_text = longest_example["answers"]["text"][0]
answer_start = longest_example["answers"]["answer_start"][0]
answer_end = answer_start + len(answer_text)

print("\n[5] Answer span")
print(f"Answer       : {answer_text}")
print(f"Character span: {answer_start}:{answer_end}")

answer_found_windows = []

for window_idx in range(num_windows):
    offsets = tokenized_long["offset_mapping"][window_idx]
    sequence_ids = tokenized_long.sequence_ids(window_idx)

    context_offsets = [
        (idx, offset)
        for idx, (offset, sequence_id) in enumerate(
            zip(offsets, sequence_ids)
        )
        if sequence_id == 1 and offset[1] > offset[0]
    ]

    if not context_offsets:
        continue

    context_start = context_offsets[0][1][0]
    context_end = context_offsets[-1][1][1]

    # Check whether the complete answer is inside this window.
    if answer_start >= context_start and answer_end <= context_end:
        answer_found_windows.append(window_idx)

print(f"Answer fully contained in windows: {answer_found_windows}")

if answer_found_windows:
    print("PASS — The answer is fully covered by at least one feature.")
else:
    print("WARNING — The answer is not fully covered by any feature.")

# ---------------------------------------------------------------------
# Show token count before and after overflow
# ---------------------------------------------------------------------

original_token_count = len(
    tokenizer(
        long_question,
        long_context,
        truncation=False,
        add_special_tokens=True,
    )["input_ids"]
)

print("\n[6] Tokenization comparison")
print(f"Original token count without truncation : {original_token_count:,}")
print(f"Maximum tokens per feature             : {MAX_LENGTH}")
print(f"Number of generated features/windows   : {num_windows}")

print("\n" + "=" * 70)
print("LONG-CONTEXT INSPECTION COMPLETE")
print("=" * 70)

LONG-CONTEXT HANDLING — OVERFLOW + STRIDE


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (866 > 512). Running this sequence through the model will result in indexing errors



[1] Longest training context
Question           : What is the largest hottest continuously large area   worldwide?
Context words      : 653
Context characters : 3,706

[2] Overflow information
Max sequence length : 384
Document stride     : 128
Number of windows   : 2
Window lengths      : [384, 154]

[3] Overflow mapping
Mapping length      : 2
Unique source IDs   : [0]

[4] Context coverage per window
Window  0: character range 0:1,685 (1,685 characters)
Window  1: character range 1,144:1,743 (599 characters)

[5] Answer span
Answer       : the Great Desert
Character span: 602:618
Answer fully contained in windows: [0]
PASS — The answer is fully covered by at least one feature.

[6] Tokenization comparison
Original token count without truncation : 866
Maximum tokens per feature             : 384
Number of generated features/windows   : 2

LONG-CONTEXT INSPECTION COMPLETE


In [7]:
# Task 6 — Question Answering with Transformers
# Step 7: Production-grade SQuAD preprocessing
#
# This step converts character-level answer spans into token-level
# start/end positions while correctly handling long contexts.

from datasets import DatasetDict

MAX_LENGTH = 384
DOC_STRIDE = 128

print("=" * 70)
print("SQUAD PREPROCESSING")
print("=" * 70)

print("\nConfiguration")
print(f"Model max sequence length : {MAX_LENGTH}")
print(f"Document stride           : {DOC_STRIDE}")


def prepare_train_features(examples):
    """
    Tokenize question-context pairs and create token-level
    start_positions and end_positions for extractive QA training.

    Long contexts are split into overlapping windows using stride.
    Windows that do not contain the complete answer are assigned
    the CLS token position as the target span.
    """

    questions = [
        question.strip()
        for question in examples["question"]
    ]

    tokenized_examples = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=False,
    )

    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )

    offset_mapping = tokenized_examples["offset_mapping"]

    start_positions = []
    end_positions = []

    for feature_index, offsets in enumerate(offset_mapping):

        # Map this tokenized feature back to its original SQuAD example.
        sample_index = sample_mapping[feature_index]

        answers = examples["answers"][sample_index]

        # SQuAD v1.1 training examples have at least one answer.
        answer_start = answers["answer_start"][0]
        answer_text = answers["text"][0]
        answer_end = answer_start + len(answer_text)

        # Find the CLS token.
        input_ids = tokenized_examples["input_ids"][feature_index]

        cls_token_index = input_ids.index(
            tokenizer.cls_token_id
        )

        sequence_ids = tokenized_examples.sequence_ids(
            feature_index
        )

        # Identify the context token range.
        token_start_index = 0

        while (
            token_start_index < len(sequence_ids)
            and sequence_ids[token_start_index] != 1
        ):
            token_start_index += 1

        token_end_index = len(sequence_ids) - 1

        while (
            token_end_index >= 0
            and sequence_ids[token_end_index] != 1
        ):
            token_end_index -= 1

        # If the context is empty or malformed, use CLS.
        if token_start_index > token_end_index:
            start_positions.append(cls_token_index)
            end_positions.append(cls_token_index)
            continue

        # If the answer is not fully inside this feature,
        # assign the CLS position.
        if (
            offsets[token_start_index][0] > answer_start
            or offsets[token_end_index][1] < answer_end
        ):
            start_positions.append(cls_token_index)
            end_positions.append(cls_token_index)

        else:
            # Move to the token whose span contains the
            # beginning of the answer.
            while (
                token_start_index <= token_end_index
                and offsets[token_start_index][0] <= answer_start
            ):
                token_start_index += 1

            start_positions.append(token_start_index - 1)

            # Move backward to the token whose span contains
            # the end of the answer.
            while (
                token_end_index >= 0
                and offsets[token_end_index][1] >= answer_end
            ):
                token_end_index -= 1

            end_positions.append(token_end_index + 1)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions

    return tokenized_examples


def prepare_validation_features(examples):
    """
    Tokenize validation examples into overlapping features.

    We keep example_id and offset_mapping so model predictions can
    later be mapped back to the original context for SQuAD-style
    Exact Match and F1 evaluation.
    """

    questions = [
        question.strip()
        for question in examples["question"]
    ]

    tokenized_examples = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=False,
    )

    sample_mapping = tokenized_examples.pop(
        "overflow_to_sample_mapping"
    )

    example_ids = []

    for feature_index in range(len(tokenized_examples["input_ids"])):

        sample_index = sample_mapping[feature_index]

        example_ids.append(
            examples["id"][sample_index]
        )

        sequence_ids = tokenized_examples.sequence_ids(
            feature_index
        )

        offsets = tokenized_examples["offset_mapping"][
            feature_index
        ]

        # Keep offsets only for context tokens.
        # Question and special-token offsets are removed because
        # they can never be part of the predicted answer.
        tokenized_examples["offset_mapping"][feature_index] = [
            offset if sequence_ids[k] == 1 else None
            for k, offset in enumerate(offsets)
        ]

    tokenized_examples["example_id"] = example_ids

    return tokenized_examples


# ---------------------------------------------------------------------
# Apply preprocessing
# ---------------------------------------------------------------------

print("\n[1] Preparing training features...")

train_features = squad["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=squad["train"].column_names,
    desc="Preparing training features",
)

print(f"Training examples : {len(squad['train']):,}")
print(f"Training features : {len(train_features):,}")

print("\n[2] Preparing validation features...")

validation_features = squad["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=squad["validation"].column_names,
    desc="Preparing validation features",
)

print(f"Validation examples : {len(squad['validation']):,}")
print(f"Validation features : {len(validation_features):,}")

# ---------------------------------------------------------------------
# Verify feature structure
# ---------------------------------------------------------------------

print("\n[3] Training feature columns")
print(train_features.column_names)

print("\n[4] Validation feature columns")
print(validation_features.column_names)

# ---------------------------------------------------------------------
# Inspect one processed training feature
# ---------------------------------------------------------------------

print("\n[5] Processed training feature")

sample_feature = train_features[0]

print(f"Number of tokens : {len(sample_feature['input_ids'])}")
print(f"Start position  : {sample_feature['start_positions']}")
print(f"End position    : {sample_feature['end_positions']}")

start_position = sample_feature["start_positions"]
end_position = sample_feature["end_positions"]

decoded_answer = tokenizer.decode(
    sample_feature["input_ids"][
        start_position:end_position + 1
    ],
    skip_special_tokens=True,
)

print(f"Decoded answer  : {decoded_answer!r}")

# ---------------------------------------------------------------------
# Basic sanity checks
# ---------------------------------------------------------------------

print("\n[6] Sanity checks")

invalid_training_positions = 0

for feature in train_features.select(
    range(min(1000, len(train_features)))
):
    start = feature["start_positions"]
    end = feature["end_positions"]
    sequence_length = len(feature["input_ids"])

    if not (
        0 <= start < sequence_length
        and 0 <= end < sequence_length
        and start <= end
    ):
        invalid_training_positions += 1

print(
    f"Invalid start/end positions "
    f"in first {min(1000, len(train_features)):,} features: "
    f"{invalid_training_positions}"
)

if invalid_training_positions == 0:
    print("PASS — Training span positions are valid.")

print("\n" + "=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)

SQUAD PREPROCESSING

Configuration
Model max sequence length : 384
Document stride           : 128

[1] Preparing training features...


Preparing training features:   0%|          | 0/87599 [00:00<?, ? examples/s]

Training examples : 87,599
Training features : 88,492

[2] Preparing validation features...


Preparing validation features:   0%|          | 0/10570 [00:00<?, ? examples/s]

Validation examples : 10,570
Validation features : 10,753

[3] Training feature columns
['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'start_positions', 'end_positions']

[4] Validation feature columns
['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id']

[5] Processed training feature
Number of tokens : 176
Start position  : 130
End position    : 137
Decoded answer  : 'saint bernadette soubirous'

[6] Sanity checks
Invalid start/end positions in first 1,000 features: 0
PASS — Training span positions are valid.

PREPROCESSING COMPLETE


In [12]:
import random

print("=" * 70)
print("DEEP PREPROCESSING VALIDATION")
print("=" * 70)

random.seed(42)

# ------------------------------------------------------------
# 1. Basic feature statistics
# ------------------------------------------------------------

train_start_positions = train_features["start_positions"]
train_end_positions = train_features["end_positions"]
train_input_ids = train_features["input_ids"]

print("\n[1] Training feature statistics")
print(f"Training features : {len(train_features):,}")

# ------------------------------------------------------------
# 2. Count CLS-based overflow features
# ------------------------------------------------------------

cls_feature_count = 0

for input_ids, start, end in zip(
    train_input_ids,
    train_start_positions,
    train_end_positions
):
    if (
        start == end
        and 0 <= start < len(input_ids)
        and input_ids[start] == tokenizer.cls_token_id
    ):
        cls_feature_count += 1

print(f"CLS-based features : {cls_feature_count:,}")
print(
    f"CLS-based feature ratio : "
    f"{cls_feature_count / len(train_features) * 100:.2f}%"
)

# ------------------------------------------------------------
# 3. Randomly validate decoded answer spans
# ------------------------------------------------------------

print("\n[2] Random span validation")

sample_indices = random.sample(
    range(len(train_features)),
    min(20, len(train_features))
)

valid_spans = 0
invalid_spans = 0
cls_samples = 0

for feature_idx in sample_indices:

    input_ids = train_features["input_ids"][feature_idx]
    start_pos = train_features["start_positions"][feature_idx]
    end_pos = train_features["end_positions"][feature_idx]

    # Validate token positions first
    if not (
        0 <= start_pos < len(input_ids)
        and 0 <= end_pos < len(input_ids)
        and start_pos <= end_pos
    ):
        invalid_spans += 1

        print(
            f"Feature {feature_idx:>6} | "
            f"INVALID span {start_pos}-{end_pos}"
        )
        continue

    # CLS position means the answer is outside this window
    if (
        start_pos == end_pos
        and input_ids[start_pos] == tokenizer.cls_token_id
    ):
        cls_samples += 1

        print(
            f"Feature {feature_idx:>6} | "
            f"CLS span | answer outside this window"
        )
        continue

    decoded_answer = tokenizer.decode(
        input_ids[start_pos:end_pos + 1],
        skip_special_tokens=True
    ).strip()

    if decoded_answer:
        valid_spans += 1

        print(
            f"Feature {feature_idx:>6} | "
            f"tokens {start_pos:>3}-{end_pos:<3} | "
            f"answer = '{decoded_answer}'"
        )
    else:
        invalid_spans += 1

print(f"\nValid sampled spans   : {valid_spans}")
print(f"CLS sampled spans     : {cls_samples}")
print(f"Invalid sampled spans : {invalid_spans}")

# ------------------------------------------------------------
# 4. Feature length statistics
# ------------------------------------------------------------

feature_lengths = [
    len(input_ids)
    for input_ids in train_input_ids
]

print("\n[3] Feature length statistics")
print(f"Minimum length : {min(feature_lengths)}")
print(f"Maximum length : {max(feature_lengths)}")
print(f"Average length : {sum(feature_lengths) / len(feature_lengths):.2f}")

# ------------------------------------------------------------
# 5. Verify MAX_LENGTH constraint
# ------------------------------------------------------------

oversized_features = sum(
    1
    for length in feature_lengths
    if length > MAX_LENGTH
)

print(
    f"\nFeatures exceeding MAX_LENGTH={MAX_LENGTH}: "
    f"{oversized_features}"
)

# ------------------------------------------------------------
# 6. Final validation
# ------------------------------------------------------------

print("\n[4] Final validation")

if oversized_features == 0:
    print("PASS — No training feature exceeds MAX_LENGTH.")

if invalid_spans == 0:
    print("PASS — All sampled training spans are valid.")

if valid_spans > 0:
    print("PASS — Sampled answer spans decode to non-empty text.")

if cls_feature_count > 0:
    print(
        "PASS — Overflow windows without the answer "
        "are correctly represented by the CLS position."
    )

print("\n" + "=" * 70)
print("DEEP VALIDATION COMPLETE")
print("=" * 70)

DEEP PREPROCESSING VALIDATION

[1] Training feature statistics
Training features : 88,492
CLS-based features : 767
CLS-based feature ratio : 0.87%

[2] Random span validation
Feature  83810 | tokens  17-20  | answer = '96. 4 %'
Feature  14592 | tokens  36-37  | answer = 'november 2005'
Feature   3278 | tokens  62-62  | answer = 'one'
Feature  36048 | tokens  59-61  | answer = 'the nicene'
Feature  32098 | tokens 103-103 | answer = 'contempt'
Feature  29256 | tokens 132-133 | answer = 'airplane mode'
Feature  18289 | tokens  41-47  | answer = 'government profiting on illegal drug trade'
Feature  13434 | tokens  86-90  | answer = 'peace in world war i'
Feature  71482 | tokens  39-39  | answer = 'cairo'
Feature  11395 | tokens  29-30  | answer = 'may 2003'
Feature  77397 | tokens 107-108 | answer = 'outdoor murals'
Feature  55302 | tokens  70-71  | answer = '10 %'
Feature   4165 | tokens 191-192 | answer = '72 %'
Feature   3905 | tokens  71-71  | answer = '1790'
Feature  12280 | tokens  1

In [14]:
# Task 6 — Question Answering with Transformers
# Step 8: Load and inspect the DistilBERT QA model

import torch
from transformers import AutoModelForQuestionAnswering

print("=" * 70)
print("DISTILBERT QUESTION ANSWERING MODEL")
print("=" * 70)

print("\n[1] Loading model...")

qa_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

print("Model loaded successfully.")

print("\n[2] Model information")
print(f"Model name        : {MODEL_NAME}")
print(f"Model class       : {qa_model.__class__.__name__}")

total_parameters = sum(
    parameter.numel()
    for parameter in qa_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in qa_model.parameters()
    if parameter.requires_grad
)

print("\n[3] Parameter statistics")
print(f"Total parameters      : {total_parameters:,}")
print(f"Trainable parameters  : {trainable_parameters:,}")

print("\n[4] Question Answering head")

print(f"QA head               : {qa_model.qa_outputs}")
print(f"QA output dimensions  : {qa_model.qa_outputs.out_features}")

if qa_model.qa_outputs.out_features == 2:
    print("Output meaning        : 2 logits per token")
    print("  - Start position logits")
    print("  - End position logits")
else:
    print("WARNING — Unexpected QA head output dimension.")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\n[5] Device")
print(f"Selected device       : {device}")

qa_model = qa_model.to(device)

print(f"Model device          : {next(qa_model.parameters()).device}")

print("\n[6] Forward-pass test")

test_feature = train_features[0]

test_input_ids = torch.tensor(
    [test_feature["input_ids"]],
    dtype=torch.long,
    device=device
)

test_attention_mask = torch.tensor(
    [test_feature["attention_mask"]],
    dtype=torch.long,
    device=device
)

with torch.no_grad():
    test_output = qa_model(
        input_ids=test_input_ids,
        attention_mask=test_attention_mask,
    )

print(f"Input shape          : {tuple(test_input_ids.shape)}")
print(f"Start logits shape   : {tuple(test_output.start_logits.shape)}")
print(f"End logits shape     : {tuple(test_output.end_logits.shape)}")

expected_shape = test_input_ids.shape

if (
    test_output.start_logits.shape == expected_shape
    and test_output.end_logits.shape == expected_shape
):
    print("\nPASS — QA model output shapes are correct.")
else:
    print("\nFAIL — Unexpected QA output shapes.")

print("\n" + "=" * 70)
print("MODEL INSPECTION COMPLETE")
print("=" * 70)

DISTILBERT QUESTION ANSWERING MODEL

[1] Loading model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully.

[2] Model information
Model name        : distilbert-base-uncased
Model class       : DistilBertForQuestionAnswering

[3] Parameter statistics
Total parameters      : 66,364,418
Trainable parameters  : 66,364,418

[4] Question Answering head
QA head               : Linear(in_features=768, out_features=2, bias=True)
QA output dimensions  : 2
Output meaning        : 2 logits per token
  - Start position logits
  - End position logits

[5] Device
Selected device       : cuda
Model device          : cuda:0

[6] Forward-pass test
Input shape          : (1, 176)
Start logits shape   : (1, 176)
End logits shape     : (1, 176)

PASS — QA model output shapes are correct.

MODEL INSPECTION COMPLETE


In [15]:
# Task 6 — Question Answering with Transformers
# Step 9: Verify Trainer API compatibility

import inspect
from transformers import TrainingArguments, Trainer

print("=" * 70)
print("TRAINER API COMPATIBILITY CHECK")
print("=" * 70)

print("\n[1] Transformers version")
print(f"Transformers : {__import__('transformers').__version__}")

print("\n[2] TrainingArguments signature")

training_args_signature = inspect.signature(
    TrainingArguments.__init__
)

print(training_args_signature)

print("\n[3] Trainer signature")

trainer_signature = inspect.signature(
    Trainer.__init__
)

print(trainer_signature)

print("\n[4] Important argument availability")

argument_names = set(
    training_args_signature.parameters.keys()
)

important_arguments = [
    "output_dir",
    "per_device_train_batch_size",
    "per_device_eval_batch_size",
    "gradient_accumulation_steps",
    "learning_rate",
    "weight_decay",
    "num_train_epochs",
    "warmup_steps",
    "warmup_ratio",
    "logging_steps",
    "eval_strategy",
    "evaluation_strategy",
    "save_strategy",
    "save_steps",
    "save_total_limit",
    "load_best_model_at_end",
    "metric_for_best_model",
    "greater_is_better",
    "fp16",
    "bf16",
    "report_to",
    "seed",
    "data_seed",
]

for argument in important_arguments:
    status = "AVAILABLE" if argument in argument_names else "NOT AVAILABLE"
    print(f"{argument:30} : {status}")

print("\n" + "=" * 70)
print("TRAINER API CHECK COMPLETE")
print("=" * 70)

TRAINER API COMPATIBILITY CHECK

[1] Transformers version
Transformers : 5.16.1

[2] TrainingArguments signature
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = False

In [16]:
# Task 6 — Question Answering with Transformers
# Step 10: Prepare datasets and data collator for training

from transformers import DefaultDataCollator

print("=" * 70)
print("TRAINING DATASET PREPARATION")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Keep only the columns required by the QA model during training
# ---------------------------------------------------------------------

train_columns = [
    "input_ids",
    "attention_mask",
    "start_positions",
    "end_positions",
]

train_dataset = train_features.select_columns(train_columns)

print("\n[1] Training dataset")
print(f"Original training features : {len(train_features):,}")
print(f"Trainer training columns   : {train_dataset.column_names}")

# ---------------------------------------------------------------------
# 2. Validation features
#
# We keep the full validation feature dataset because later we need:
# - input_ids
# - attention_mask
# - example_id
# - offset_mapping
#
# for SQuAD-style post-processing.
# ---------------------------------------------------------------------

print("\n[2] Validation features")
print(f"Validation features        : {len(validation_features):,}")
print(f"Validation columns         : {validation_features.column_names}")

# ---------------------------------------------------------------------
# 3. Data collator
# ---------------------------------------------------------------------

data_collator = DefaultDataCollator()

print("\n[3] Data collator")
print(f"Collator class             : {data_collator.__class__.__name__}")

# ---------------------------------------------------------------------
# 4. Inspect one training example
# ---------------------------------------------------------------------

print("\n[4] Sample training feature")

sample = train_dataset[0]

for key, value in sample.items():
    if isinstance(value, list):
        print(f"{key:<18}: list length = {len(value)}")
    else:
        print(f"{key:<18}: {value}")

# ---------------------------------------------------------------------
# 5. Verify label positions
# ---------------------------------------------------------------------

start_position = sample["start_positions"]
end_position = sample["end_positions"]
sequence_length = len(sample["input_ids"])

print("\n[5] Label validation")
print(f"Sequence length : {sequence_length}")
print(f"Start position  : {start_position}")
print(f"End position    : {end_position}")

if (
    0 <= start_position < sequence_length
    and 0 <= end_position < sequence_length
    and start_position <= end_position
):
    print("PASS — Sample start/end positions are valid.")
else:
    print("FAIL — Invalid sample start/end positions.")

# ---------------------------------------------------------------------
# 6. Verify all training labels
# ---------------------------------------------------------------------

print("\n[6] Full training label validation")

invalid_labels = 0

for feature in train_dataset:
    sequence_length = len(feature["input_ids"])
    start = feature["start_positions"]
    end = feature["end_positions"]

    if not (
        0 <= start < sequence_length
        and 0 <= end < sequence_length
        and start <= end
    ):
        invalid_labels += 1

print(f"Invalid training labels : {invalid_labels:,}")

if invalid_labels == 0:
    print("PASS — All training labels are valid.")
else:
    print("FAIL — Invalid training labels detected.")

print("\n" + "=" * 70)
print("TRAINING DATASET PREPARATION COMPLETE")
print("=" * 70)

TRAINING DATASET PREPARATION

[1] Training dataset
Original training features : 88,492
Trainer training columns   : ['input_ids', 'attention_mask', 'start_positions', 'end_positions']

[2] Validation features
Validation features        : 10,753
Validation columns         : ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id']

[3] Data collator
Collator class             : DefaultDataCollator

[4] Sample training feature
input_ids         : list length = 176
attention_mask    : list length = 176
start_positions   : 130
end_positions     : 137

[5] Label validation
Sequence length : 176
Start position  : 130
End position    : 137
PASS — Sample start/end positions are valid.

[6] Full training label validation
Invalid training labels : 0
PASS — All training labels are valid.

TRAINING DATASET PREPARATION COMPLETE


In [17]:
# Task 6 — Question Answering with Transformers
# Step 11: Validate dynamic padding for variable-length QA features

import torch
from transformers import DataCollatorWithPadding

print("=" * 70)
print("DYNAMIC PADDING VALIDATION")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Create a dynamic padding collator
# ---------------------------------------------------------------------

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

print("\n[1] Data collator")
print(f"Collator class : {data_collator.__class__.__name__}")
print("Padding mode   : dynamic / longest sequence in each batch")

# ---------------------------------------------------------------------
# 2. Select features with different sequence lengths
# ---------------------------------------------------------------------

short_feature = train_dataset[0]

different_length_index = None

for index in range(1, len(train_dataset)):
    if len(train_dataset[index]["input_ids"]) != len(
        short_feature["input_ids"]
    ):
        different_length_index = index
        break

if different_length_index is None:
    raise RuntimeError(
        "Could not find two training features with different lengths."
    )

longer_feature = train_dataset[different_length_index]

print("\n[2] Test features")
print(f"Feature 0 length       : {len(short_feature['input_ids'])}")
print(
    f"Feature {different_length_index} length : "
    f"{len(longer_feature['input_ids'])}"
)

# ---------------------------------------------------------------------
# 3. Build a test batch
# ---------------------------------------------------------------------

test_batch = data_collator(
    [short_feature, longer_feature]
)

print("\n[3] Batched tensors")

for key, value in test_batch.items():
    if isinstance(value, torch.Tensor):
        print(f"{key:<18}: shape = {tuple(value.shape)}")
    else:
        print(f"{key:<18}: type = {type(value).__name__}")

# ---------------------------------------------------------------------
# 4. Validate batch dimensions
# ---------------------------------------------------------------------

batch_size = 2
expected_sequence_length = max(
    len(short_feature["input_ids"]),
    len(longer_feature["input_ids"]),
)

print("\n[4] Batch validation")
print(f"Expected batch size       : {batch_size}")
print(f"Expected padded length    : {expected_sequence_length}")

input_shape = tuple(test_batch["input_ids"].shape)
attention_shape = tuple(test_batch["attention_mask"].shape)

print(f"Actual input_ids shape    : {input_shape}")
print(f"Actual attention shape    : {attention_shape}")

if input_shape == (batch_size, expected_sequence_length):
    print("PASS — input_ids were dynamically padded correctly.")
else:
    print("FAIL — Unexpected input_ids shape.")

if attention_shape == (batch_size, expected_sequence_length):
    print("PASS — attention_mask shape is correct.")
else:
    print("FAIL — Unexpected attention_mask shape.")

# ---------------------------------------------------------------------
# 5. Validate QA labels
# ---------------------------------------------------------------------

print("\n[5] QA label validation")

print(
    f"start_positions shape     : "
    f"{tuple(test_batch['start_positions'].shape)}"
)

print(
    f"end_positions shape       : "
    f"{tuple(test_batch['end_positions'].shape)}"
)

if (
    test_batch["start_positions"].shape == (batch_size,)
    and test_batch["end_positions"].shape == (batch_size,)
):
    print("PASS — Start/end labels were batched correctly.")
else:
    print("FAIL — Unexpected label shapes.")

print("\n" + "=" * 70)
print("DYNAMIC PADDING VALIDATION COMPLETE")
print("=" * 70)

DYNAMIC PADDING VALIDATION

[1] Data collator
Collator class : DataCollatorWithPadding
Padding mode   : dynamic / longest sequence in each batch

[2] Test features
Feature 0 length       : 176
Feature 1 length : 172

[3] Batched tensors
input_ids         : shape = (2, 176)
attention_mask    : shape = (2, 176)
start_positions   : shape = (2,)
end_positions     : shape = (2,)

[4] Batch validation
Expected batch size       : 2
Expected padded length    : 176
Actual input_ids shape    : (2, 176)
Actual attention shape    : (2, 176)
PASS — input_ids were dynamically padded correctly.
PASS — attention_mask shape is correct.

[5] QA label validation
start_positions shape     : (2,)
end_positions shape       : (2,)
PASS — Start/end labels were batched correctly.

DYNAMIC PADDING VALIDATION COMPLETE


In [18]:
# Task 6 — Question Answering with Transformers
# Step 11: Configure Trainer for DistilBERT SQuAD fine-tuning

import os
import torch
from transformers import TrainingArguments, Trainer

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Verify GPU
# ---------------------------------------------------------------------

print("\n[1] Hardware")

if torch.cuda.is_available():
    print(f"Device : {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory : "
        f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
    )
else:
    raise RuntimeError("CUDA GPU is required for the planned training run.")

# ---------------------------------------------------------------------
# 2. Training configuration
# ---------------------------------------------------------------------

OUTPUT_DIR = "/content/qa_distilbert_squad"
NUM_EPOCHS = 2
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 500
LOGGING_STEPS = 100
SEED = 42

print("\n[2] Training hyperparameters")

print(f"Output directory            : {OUTPUT_DIR}")
print(f"Epochs                      : {NUM_EPOCHS}")
print(f"Train batch size            : {TRAIN_BATCH_SIZE}")
print(f"Eval batch size             : {EVAL_BATCH_SIZE}")
print(f"Gradient accumulation      : {GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate               : {LEARNING_RATE}")
print(f"Weight decay                : {WEIGHT_DECAY}")
print(f"Warmup steps                : {WARMUP_STEPS}")
print(f"Logging steps               : {LOGGING_STEPS}")
print(f"Seed                        : {SEED}")
print(f"FP16                        : True")

# ---------------------------------------------------------------------
# 3. Create output directory
# ---------------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------
# 4. TrainingArguments
#
# Validation is intentionally disabled during Trainer.train().
# SQuAD evaluation requires post-processing:
#
# start/end logits
#       ↓
# feature offsets
#       ↓
# original context
#       ↓
# predicted answer text
#       ↓
# Exact Match + F1
#
# We will perform this correctly after training.
# ---------------------------------------------------------------------

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    num_train_epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,

    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,

    eval_strategy="no",

    save_strategy="epoch",
    save_total_limit=2,

    fp16=True,
    bf16=False,

    seed=SEED,
    data_seed=SEED,

    report_to="none",

    remove_unused_columns=True,
)

print("\n[3] TrainingArguments")
print("TrainingArguments created successfully.")

# ---------------------------------------------------------------------
# 5. Create Trainer
# ---------------------------------------------------------------------

trainer = Trainer(
    model=qa_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("\n[4] Trainer")
print(f"Trainer class : {trainer.__class__.__name__}")
print(f"Training rows : {len(train_dataset):,}")

# ---------------------------------------------------------------------
# 6. Calculate expected optimization steps
# ---------------------------------------------------------------------

steps_per_epoch = (
    len(train_dataset)
    + TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
    - 1
) // (
    TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
)

total_training_steps = steps_per_epoch * NUM_EPOCHS

print("\n[5] Training step estimate")
print(f"Steps per epoch      : {steps_per_epoch:,}")
print(f"Total training steps : {total_training_steps:,}")

# ---------------------------------------------------------------------
# 7. Final configuration validation
# ---------------------------------------------------------------------

print("\n[6] Configuration validation")

if training_args.fp16 and torch.cuda.is_available():
    print("PASS — FP16 training is enabled on CUDA.")

if training_args.eval_strategy == "no":
    print(
        "PASS — In-training evaluation disabled; "
        "SQuAD post-processing will be performed after training."
    )

if training_args.save_strategy == "epoch":
    print("PASS — Checkpoints will be saved after each epoch.")

if training_args.seed == 42 and training_args.data_seed == 42:
    print("PASS — Reproducibility seeds are fixed.")

print("\n" + "=" * 70)
print("TRAINING CONFIGURATION COMPLETE")
print("=" * 70)

TRAINING CONFIGURATION

[1] Hardware
Device : Tesla T4
GPU memory : 14.56 GB

[2] Training hyperparameters
Output directory            : /content/qa_distilbert_squad
Epochs                      : 2
Train batch size            : 8
Eval batch size             : 8
Gradient accumulation      : 1
Learning rate               : 3e-05
Weight decay                : 0.01
Warmup steps                : 500
Logging steps               : 100
Seed                        : 42
FP16                        : True

[3] TrainingArguments
TrainingArguments created successfully.

[4] Trainer
Trainer class : Trainer
Training rows : 88,492

[5] Training step estimate
Steps per epoch      : 11,062
Total training steps : 22,124

[6] Configuration validation
PASS — FP16 training is enabled on CUDA.
PASS — In-training evaluation disabled; SQuAD post-processing will be performed after training.
PASS — Checkpoints will be saved after each epoch.
PASS — Reproducibility seeds are fixed.

TRAINING CONFIGURATION COMPLET

In [19]:
# Task 6 — Question Answering with Transformers
# Step 12: Trainer smoke test before full training

import torch
import time

print("=" * 70)
print("TRAINING SMOKE TEST")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Verify Trainer components
# ---------------------------------------------------------------------

print("\n[1] Trainer components")

print(f"Model class    : {trainer.model.__class__.__name__}")
print(f"Dataset size   : {len(trainer.train_dataset):,}")
print(f"Collator       : {trainer.data_collator.__class__.__name__}")
print(f"Device         : {trainer.args.device}")
print(f"FP16           : {trainer.args.fp16}")

# ---------------------------------------------------------------------
# 2. Build one real training batch
# ---------------------------------------------------------------------

print("\n[2] Building one real training batch")

batch_features = [
    train_dataset[i]
    for i in range(8)
]

batch = data_collator(batch_features)

print(f"Batch size     : {batch['input_ids'].shape[0]}")
print(f"Sequence length: {batch['input_ids'].shape[1]}")
print(f"Input shape    : {tuple(batch['input_ids'].shape)}")
print(f"Attention shape: {tuple(batch['attention_mask'].shape)}")
print(f"Start labels   : {tuple(batch['start_positions'].shape)}")
print(f"End labels     : {tuple(batch['end_positions'].shape)}")

# ---------------------------------------------------------------------
# 3. Move batch to GPU
# ---------------------------------------------------------------------

device = trainer.args.device

batch = {
    key: value.to(device)
    for key, value in batch.items()
}

print("\n[3] Batch device")

for key, value in batch.items():
    print(f"{key:<18}: {value.device}")

# ---------------------------------------------------------------------
# 4. Forward pass with labels
# ---------------------------------------------------------------------

print("\n[4] Forward pass")

trainer.model.train()

start_time = time.time()

outputs = trainer.model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    start_positions=batch["start_positions"],
    end_positions=batch["end_positions"],
)

forward_time = time.time() - start_time

print(f"Loss          : {outputs.loss.item():.6f}")
print(f"Start logits  : {tuple(outputs.start_logits.shape)}")
print(f"End logits    : {tuple(outputs.end_logits.shape)}")
print(f"Forward time  : {forward_time:.3f} seconds")

# ---------------------------------------------------------------------
# 5. Backward pass
# ---------------------------------------------------------------------

print("\n[5] Backward pass")

trainer.model.zero_grad(set_to_none=True)

outputs.loss.backward()

gradient_tensors = 0
nonzero_gradient_tensors = 0

for parameter in trainer.model.parameters():
    if parameter.grad is not None:
        gradient_tensors += 1

        if torch.any(parameter.grad != 0):
            nonzero_gradient_tensors += 1

print(f"Parameters with gradients       : {gradient_tensors}")
print(f"Parameters with non-zero gradients: {nonzero_gradient_tensors}")

# ---------------------------------------------------------------------
# 6. Final validation
# ---------------------------------------------------------------------

print("\n[6] Smoke test validation")

if torch.isfinite(outputs.loss):
    print("PASS — Training loss is finite.")
else:
    print("FAIL — Training loss is NaN or infinite.")

if (
    outputs.start_logits.shape == batch["input_ids"].shape
    and outputs.end_logits.shape == batch["input_ids"].shape
):
    print("PASS — Start/end logits have correct shapes.")
else:
    print("FAIL — Unexpected logits shape.")

if gradient_tensors > 0 and nonzero_gradient_tensors > 0:
    print("PASS — Backward pass produced non-zero gradients.")
else:
    print("FAIL — No valid gradients were produced.")

# ---------------------------------------------------------------------
# 7. Clear smoke-test gradients
# ---------------------------------------------------------------------

trainer.model.zero_grad(set_to_none=True)

print("\nSmoke-test gradients cleared.")

print("\n" + "=" * 70)
print("TRAINING SMOKE TEST COMPLETE")
print("=" * 70)

TRAINING SMOKE TEST

[1] Trainer components
Model class    : DistilBertForQuestionAnswering
Dataset size   : 88,492
Collator       : DataCollatorWithPadding
Device         : cuda:0
FP16           : True

[2] Building one real training batch
Batch size     : 8
Sequence length: 271
Input shape    : (8, 271)
Attention shape: (8, 271)
Start labels   : (8,)
End labels     : (8,)

[3] Batch device
input_ids         : cuda:0
attention_mask    : cuda:0
start_positions   : cuda:0
end_positions     : cuda:0

[4] Forward pass
Loss          : 5.731484
Start logits  : (8, 271)
End logits    : (8, 271)
Forward time  : 0.537 seconds

[5] Backward pass
Parameters with gradients       : 102
Parameters with non-zero gradients: 102

[6] Smoke test validation
PASS — Training loss is finite.
PASS — Start/end logits have correct shapes.
PASS — Backward pass produced non-zero gradients.

Smoke-test gradients cleared.

TRAINING SMOKE TEST COMPLETE


In [20]:
# Task 6 — Question Answering with Transformers
# Step 13: Full DistilBERT fine-tuning on SQuAD v1.1

import time
import torch

print("=" * 70)
print("STARTING FULL SQUAD FINE-TUNING")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Final pre-training information
# ---------------------------------------------------------------------

print("\n[1] Training overview")

print(f"Model              : {MODEL_NAME}")
print(f"Training features  : {len(train_dataset):,}")
print(f"Epochs             : {NUM_EPOCHS}")
print(f"Batch size         : {TRAIN_BATCH_SIZE}")
print(f"Learning rate      : {LEARNING_RATE}")
print(f"Max sequence length: {MAX_LENGTH}")
print(f"Document stride    : {DOC_STRIDE}")
print(f"FP16               : {training_args.fp16}")
print(f"Expected steps     : {total_training_steps:,}")
print(f"Output directory   : {OUTPUT_DIR}")

# ---------------------------------------------------------------------
# 2. GPU memory before training
# ---------------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.empty_cache()

    allocated_before = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved_before = torch.cuda.memory_reserved() / (1024 ** 3)

    print("\n[2] GPU memory before training")
    print(f"Allocated : {allocated_before:.2f} GB")
    print(f"Reserved  : {reserved_before:.2f} GB")

# ---------------------------------------------------------------------
# 3. Start training
# ---------------------------------------------------------------------

print("\n[3] Starting Trainer.train()")
print("-" * 70)

training_start_time = time.time()

train_result = trainer.train()

training_elapsed = time.time() - training_start_time

# ---------------------------------------------------------------------
# 4. Training completed
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print("\n[4] Training time")

hours = int(training_elapsed // 3600)
minutes = int((training_elapsed % 3600) // 60)
seconds = int(training_elapsed % 60)

print(
    f"Elapsed time : "
    f"{hours:02d}:{minutes:02d}:{seconds:02d}"
)

print(f"Elapsed seconds: {training_elapsed:.2f}")

# ---------------------------------------------------------------------
# 5. Training metrics
# ---------------------------------------------------------------------

print("\n[5] Trainer metrics")

for key, value in train_result.metrics.items():
    if isinstance(value, (int, float)):
        print(f"{key:<25}: {value}")

# ---------------------------------------------------------------------
# 6. GPU memory after training
# ---------------------------------------------------------------------

if torch.cuda.is_available():
    allocated_after = torch.cuda.memory_allocated() / (1024 ** 3)
    reserved_after = torch.cuda.memory_reserved() / (1024 ** 3)

    print("\n[6] GPU memory after training")
    print(f"Allocated : {allocated_after:.2f} GB")
    print(f"Reserved  : {reserved_after:.2f} GB")

print("\n" + "=" * 70)
print("FULL FINE-TUNING COMPLETE")
print("=" * 70)

STARTING FULL SQUAD FINE-TUNING

[1] Training overview
Model              : distilbert-base-uncased
Training features  : 88,492
Epochs             : 2
Batch size         : 8
Learning rate      : 3e-05
Max sequence length: 384
Document stride    : 128
FP16               : True
Expected steps     : 22,124
Output directory   : /content/qa_distilbert_squad

[2] GPU memory before training
Allocated : 0.27 GB
Reserved  : 0.31 GB

[3] Starting Trainer.train()
----------------------------------------------------------------------


Step,Training Loss
100,5.537600
200,4.635707
300,3.848759
400,3.616306
500,2.842370
600,2.568132
700,2.218312
800,2.033144
900,1.905892
1000,1.901189


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


TRAINING COMPLETED

[4] Training time
Elapsed time : 00:25:45
Elapsed seconds: 1545.13

[5] Trainer metrics
train_runtime            : 1543.8882
train_samples_per_second : 114.635
train_steps_per_second   : 14.33
total_flos               : 1.228545900147384e+16
train_loss               : 1.1720492330732881
epoch                    : 2.0

[6] GPU memory after training
Allocated : 0.76 GB
Reserved  : 1.67 GB

FULL FINE-TUNING COMPLETE


In [21]:
# Task 6 — Question Answering with Transformers
# Step 14: Save final fine-tuned model and tokenizer

import os
import torch

FINAL_MODEL_DIR = "/content/qa_distilbert_squad/final_model"

print("=" * 70)
print("SAVING FINAL FINE-TUNED MODEL")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Create final model directory
# ---------------------------------------------------------------------

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# ---------------------------------------------------------------------
# 2. Save model
# ---------------------------------------------------------------------

trainer.save_model(FINAL_MODEL_DIR)

# ---------------------------------------------------------------------
# 3. Save tokenizer
# ---------------------------------------------------------------------

tokenizer.save_pretrained(FINAL_MODEL_DIR)

# ---------------------------------------------------------------------
# 4. Verify saved files
# ---------------------------------------------------------------------

print("\n[1] Saved model directory")
print(FINAL_MODEL_DIR)

print("\n[2] Saved files")

for filename in sorted(os.listdir(FINAL_MODEL_DIR)):
    filepath = os.path.join(FINAL_MODEL_DIR, filename)

    if os.path.isfile(filepath):
        size_mb = os.path.getsize(filepath) / (1024 ** 2)
        print(f"{filename:<45} {size_mb:>8.2f} MB")

# ---------------------------------------------------------------------
# 5. Check required files
# ---------------------------------------------------------------------

required_files = [
    "config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
]

model_files = [
    filename
    for filename in os.listdir(FINAL_MODEL_DIR)
    if filename.endswith(".safetensors") or filename.endswith(".bin")
]

print("\n[3] Save validation")

missing_files = [
    filename
    for filename in required_files
    if not os.path.exists(os.path.join(FINAL_MODEL_DIR, filename))
]

if not missing_files and len(model_files) > 0:
    print("PASS — Final model and tokenizer files were saved successfully.")
else:
    print("FAIL — Some expected files are missing.")
    print(f"Missing required files: {missing_files}")
    print(f"Model weight files found: {model_files}")

print("\n" + "=" * 70)
print("FINAL MODEL SAVE COMPLETE")
print("=" * 70)

SAVING FINAL FINE-TUNED MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[1] Saved model directory
/content/qa_distilbert_squad/final_model

[2] Saved files
config.json                                       0.00 MB
model.safetensors                               253.17 MB
tokenizer.json                                    0.68 MB
tokenizer_config.json                             0.00 MB
training_args.bin                                 0.00 MB

[3] Save validation
FAIL — Some expected files are missing.
Missing required files: ['special_tokens_map.json']
Model weight files found: ['model.safetensors', 'training_args.bin']

FINAL MODEL SAVE COMPLETE


In [22]:
# Task 6 — Question Answering with Transformers
# Step 15: Verify saved model and tokenizer can be reloaded

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
)

print("=" * 70)
print("SAVED MODEL RELOAD VERIFICATION")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Reload tokenizer
# ---------------------------------------------------------------------

print("\n[1] Loading tokenizer...")

saved_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

print(f"Tokenizer class : {saved_tokenizer.__class__.__name__}")
print(f"Vocab size      : {saved_tokenizer.vocab_size}")
print(f"Fast tokenizer  : {saved_tokenizer.is_fast}")

# ---------------------------------------------------------------------
# 2. Reload model
# ---------------------------------------------------------------------

print("\n[2] Loading model...")

saved_model = AutoModelForQuestionAnswering.from_pretrained(
    FINAL_MODEL_DIR
)

saved_model = saved_model.to(trainer.args.device)

print(f"Model class     : {saved_model.__class__.__name__}")
print(f"Device          : {saved_model.device}")

# ---------------------------------------------------------------------
# 3. Run a real inference test
# ---------------------------------------------------------------------

print("\n[3] Running reload inference test...")

test_question = (
    "To whom did the Virgin Mary allegedly appear "
    "in 1858 in Lourdes France?"
)

test_context = (
    "Architectural historians often study Notre Dame because of "
    "its Gothic architecture. The Virgin Mary reputedly appeared "
    "to Saint Bernadette Soubirous in 1858 in Lourdes, France."
)

test_inputs = saved_tokenizer(
    test_question,
    test_context,
    return_tensors="pt",
    truncation="only_second",
    max_length=MAX_LENGTH,
)

test_inputs = {
    key: value.to(trainer.args.device)
    for key, value in test_inputs.items()
}

saved_model.eval()

with torch.no_grad():
    test_outputs = saved_model(**test_inputs)

print(f"Input shape     : {tuple(test_inputs['input_ids'].shape)}")
print(f"Start logits    : {tuple(test_outputs.start_logits.shape)}")
print(f"End logits      : {tuple(test_outputs.end_logits.shape)}")

# ---------------------------------------------------------------------
# 4. Basic validation
# ---------------------------------------------------------------------

if (
    test_outputs.start_logits.shape
    == test_inputs["input_ids"].shape
    and
    test_outputs.end_logits.shape
    == test_inputs["input_ids"].shape
):
    print("\nPASS — Saved model successfully reloaded and produced valid QA logits.")
else:
    print("\nFAIL — Reloaded model produced unexpected output shapes.")

print("\n" + "=" * 70)
print("RELOAD VERIFICATION COMPLETE")
print("=" * 70)

SAVED MODEL RELOAD VERIFICATION

[1] Loading tokenizer...
Tokenizer class : BertTokenizer
Vocab size      : 30522
Fast tokenizer  : True

[2] Loading model...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model class     : DistilBertForQuestionAnswering
Device          : cuda:0

[3] Running reload inference test...
Input shape     : (1, 53)
Start logits    : (1, 53)
End logits      : (1, 53)

PASS — Saved model successfully reloaded and produced valid QA logits.

RELOAD VERIFICATION COMPLETE


In [25]:
# Task 6 — Question Answering with Transformers
# Step 16: Generate validation logits

import numpy as np
import torch
import time
from transformers import Trainer

print("=" * 70)
print("GENERATING VALIDATION PREDICTIONS")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Prepare validation features for model inference
# ---------------------------------------------------------------------

validation_model_columns = [
    "input_ids",
    "attention_mask",
]

validation_model_features = validation_features.select_columns(
    validation_model_columns
)

print("\n[1] Validation features")
print(f"Total validation features : {len(validation_model_features):,}")
print(f"Model input columns       : {validation_model_columns}")

# ---------------------------------------------------------------------
# 2. Create prediction trainer
# ---------------------------------------------------------------------

prediction_trainer = Trainer(
    model=saved_model,
    args=training_args,
    data_collator=data_collator,
)

print(f"Prediction model          : {saved_model.__class__.__name__}")
print(f"Prediction device         : {saved_model.device}")

# ---------------------------------------------------------------------
# 3. Generate logits
# ---------------------------------------------------------------------

print("\n[2] Running model prediction...")
print("This may take a few minutes.")

prediction_start_time = time.time()

prediction_output = prediction_trainer.predict(
    validation_model_features
)

prediction_elapsed = time.time() - prediction_start_time

# ---------------------------------------------------------------------
# 4. Extract logits
# ---------------------------------------------------------------------

start_logits = prediction_output.predictions[0]
end_logits = prediction_output.predictions[1]

print("\n[3] Prediction output")

print(f"Start logits shape : {start_logits.shape}")
print(f"End logits shape   : {end_logits.shape}")
print(f"Prediction time    : {prediction_elapsed:.2f} seconds")

# ---------------------------------------------------------------------
# 5. Basic validation
# ---------------------------------------------------------------------

expected_features = len(validation_features)

if (
    start_logits.shape[0] == expected_features
    and end_logits.shape[0] == expected_features
):
    print("\nPASS — Logits generated for every validation feature.")
else:
    print("\nFAIL — Validation feature count does not match logits.")

if start_logits.shape[1] == end_logits.shape[1]:
    print("PASS — Start/end logits have matching sequence dimensions.")
else:
    print("FAIL — Start/end logits dimensions do not match.")

print("\n" + "=" * 70)
print("VALIDATION LOGIT GENERATION COMPLETE")
print("=" * 70)

GENERATING VALIDATION PREDICTIONS

[1] Validation features
Total validation features : 10,753
Model input columns       : ['input_ids', 'attention_mask']
Prediction model          : DistilBertForQuestionAnswering
Prediction device         : cuda:0

[2] Running model prediction...
This may take a few minutes.



[3] Prediction output
Start logits shape : (10753, 384)
End logits shape   : (10753, 384)
Prediction time    : 23.63 seconds

PASS — Logits generated for every validation feature.
PASS — Start/end logits have matching sequence dimensions.

VALIDATION LOGIT GENERATION COMPLETE


In [27]:
# Task 6 — Question Answering with Transformers
# Step 17: Fast validation answer-span decoding

import numpy as np
import pandas as pd
import time

print("=" * 70)
print("FAST VALIDATION ANSWER-SPAN DECODING")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------------------

N_BEST = 20
MAX_ANSWER_LENGTH = 30

print("\n[1] Decoding configuration")
print(f"Validation features : {len(validation_features):,}")
print(f"N-best candidates   : {N_BEST}")
print(f"Max answer tokens   : {MAX_ANSWER_LENGTH}")

# ---------------------------------------------------------------------
# 2. Recover original validation examples efficiently
# ---------------------------------------------------------------------

print("\n[2] Building example lookup")

validation_examples = None

# Check common variable names that may exist in the notebook.
for variable_name in [
    "dataset",
    "squad",
    "squad_data",
    "raw_datasets",
]:

    if variable_name in globals():

        dataset_object = globals()[variable_name]

        try:
            validation_examples = dataset_object["validation"]
            break
        except Exception:
            continue

if validation_examples is None:
    raise RuntimeError(
        "Could not find the original SQuAD validation dataset "
        "in the current notebook state."
    )

# Convert the complete validation split to a dictionary once.
example_lookup = {
    example["id"]: example
    for example in validation_examples
}

print(
    f"Original validation examples : "
    f"{len(example_lookup):,}"
)

# ---------------------------------------------------------------------
# 3. Group validation features by original example
# ---------------------------------------------------------------------

example_to_features = {}

for feature_index, example_id in enumerate(
    validation_features["example_id"]
):
    example_to_features.setdefault(
        example_id,
        []
    ).append(feature_index)

print(
    f"Examples with features        : "
    f"{len(example_to_features):,}"
)

# ---------------------------------------------------------------------
# 4. Decode answer spans
# ---------------------------------------------------------------------

print("\n[3] Decoding answer spans...")

decoding_start_time = time.time()

predictions = []

for example_id, feature_indices in example_to_features.items():

    example = example_lookup[example_id]
    context = example["context"]

    feature_candidates = []

    # ---------------------------------------------------------------
    # Decode each overflow feature
    # ---------------------------------------------------------------

    for feature_index in feature_indices:

        start_logit = start_logits[feature_index]
        end_logit = end_logits[feature_index]

        offsets = validation_features["offset_mapping"][
            feature_index
        ]

        # Top start/end positions
        start_indexes = np.argsort(
            start_logit
        )[-N_BEST:][::-1]

        end_indexes = np.argsort(
            end_logit
        )[-N_BEST:][::-1]

        # -----------------------------------------------------------
        # Generate valid start/end combinations
        # -----------------------------------------------------------

        for start_index in start_indexes:

            start_offset = offsets[start_index]

            if start_offset is None:
                continue

            start_char = start_offset[0]

            for end_index in end_indexes:

                # Start must occur before or at end.
                if start_index > end_index:
                    continue

                # Limit answer length.
                if (
                    end_index - start_index + 1
                    > MAX_ANSWER_LENGTH
                ):
                    continue

                end_offset = offsets[end_index]

                if end_offset is None:
                    continue

                end_char = end_offset[1]

                if start_char >= end_char:
                    continue

                if start_char < 0:
                    continue

                if end_char > len(context):
                    continue

                answer_text = context[
                    start_char:end_char
                ].strip()

                if not answer_text:
                    continue

                score = (
                    float(start_logit[start_index])
                    + float(end_logit[end_index])
                )

                feature_candidates.append(
                    (
                        score,
                        answer_text,
                        feature_index,
                        start_index,
                        end_index,
                    )
                )

    # ---------------------------------------------------------------
    # Select best candidate across all features
    # ---------------------------------------------------------------

    if feature_candidates:

        best_candidate = max(
            feature_candidates,
            key=lambda item: item[0]
        )

        score = best_candidate[0]
        answer_text = best_candidate[1]

    else:

        score = float("-inf")
        answer_text = ""

    predictions.append(
        {
            "id": example_id,
            "prediction_text": answer_text,
            "score": score,
        }
    )

# ---------------------------------------------------------------------
# 5. Convert to DataFrame
# ---------------------------------------------------------------------

predictions_df = pd.DataFrame(predictions)

decoding_elapsed = (
    time.time() - decoding_start_time
)

print("\n[4] Decoding results")

print(
    f"Predicted examples : "
    f"{len(predictions_df):,}"
)

print(
    f"Non-empty answers  : "
    f"{(predictions_df['prediction_text'].str.len() > 0).sum():,}"
)

print(
    f"Empty answers      : "
    f"{(predictions_df['prediction_text'].str.len() == 0).sum():,}"
)

print(
    f"Decoding time      : "
    f"{decoding_elapsed:.2f} seconds"
)

# ---------------------------------------------------------------------
# 6. Inspect predictions
# ---------------------------------------------------------------------

print("\n[5] Sample predictions")

display(
    predictions_df.head(10)
)

# ---------------------------------------------------------------------
# 7. Final validation
# ---------------------------------------------------------------------

if len(predictions_df) == len(example_lookup):
    print(
        "\nPASS — One prediction generated "
        "for every validation example."
    )
else:
    print(
        "\nFAIL — Prediction count does not match "
        "the validation examples."
    )

print("\n" + "=" * 70)
print("FAST ANSWER-SPAN DECODING COMPLETE")
print("=" * 70)

FAST VALIDATION ANSWER-SPAN DECODING

[1] Decoding configuration
Validation features : 10,753
N-best candidates   : 20
Max answer tokens   : 30

[2] Building example lookup
Original validation examples : 10,570
Examples with features        : 10,570

[3] Decoding answer spans...

[4] Decoding results
Predicted examples : 10,570
Non-empty answers  : 10,570
Empty answers      : 0
Decoding time      : 25.71 seconds

[5] Sample predictions


,id,prediction_text,score
0,56be4db0acb8001400a502ec,Denver Broncos,15.769531
1,56be4db0acb8001400a502ed,Carolina Panthers,15.714844
2,56be4db0acb8001400a502ee,Levi's Stadium in the San Francisco Bay Area,3.631836
3,56be4db0acb8001400a502ef,Denver Broncos,7.449219
4,56be4db0acb8001400a502f0,gold,11.488281
5,56be8e613aeaaa14008c90d1,"""golden anniversary""",3.833984
6,56be8e613aeaaa14008c90d2,"February 7, 2016",12.871094
7,56be8e613aeaaa14008c90d3,American Football Conference,16.066406
8,56bea9923aeaaa14008c91b9,"""golden anniversary""",3.833984
9,56bea9923aeaaa14008c91ba,American Football Conference,18.593750



PASS — One prediction generated for every validation example.

FAST ANSWER-SPAN DECODING COMPLETE


In [28]:
# Task 6 — Question Answering with Transformers
# Step 18: Official SQuAD EM/F1 Evaluation

import evaluate
import pandas as pd

print("=" * 70)
print("OFFICIAL SQuAD EVALUATION")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Load SQuAD evaluation metric
# ---------------------------------------------------------------------

print("\n[1] Loading SQuAD evaluation metric...")

squad_metric = evaluate.load("squad")

print("Metric loaded successfully.")

# ---------------------------------------------------------------------
# 2. Build references from the original validation set
# ---------------------------------------------------------------------

print("\n[2] Building ground-truth references...")

references = [
    {
        "id": example["id"],
        "answers": example["answers"],
    }
    for example in validation_examples
]

print(f"References : {len(references):,}")

# ---------------------------------------------------------------------
# 3. Prepare predictions
# ---------------------------------------------------------------------

predictions = [
    {
        "id": row["id"],
        "prediction_text": row["prediction_text"],
    }
    for _, row in predictions_df.iterrows()
]

print(f"Predictions : {len(predictions):,}")

# ---------------------------------------------------------------------
# 4. Validate IDs
# ---------------------------------------------------------------------

reference_ids = {item["id"] for item in references}
prediction_ids = {item["id"] for item in predictions}

missing_predictions = reference_ids - prediction_ids
extra_predictions = prediction_ids - reference_ids

print("\n[3] ID validation")

print(f"Missing predictions : {len(missing_predictions):,}")
print(f"Extra predictions   : {len(extra_predictions):,}")

if missing_predictions:
    raise ValueError(
        "Some validation examples do not have predictions."
    )

if extra_predictions:
    raise ValueError(
        "Predictions contain unknown example IDs."
    )

print("PASS — Prediction/reference IDs match.")

# ---------------------------------------------------------------------
# 5. Compute official SQuAD metrics
# ---------------------------------------------------------------------

print("\n[4] Computing EM and F1...")

results = squad_metric.compute(
    predictions=predictions,
    references=references,
)

# ---------------------------------------------------------------------
# 6. Display results
# ---------------------------------------------------------------------

exact_match = results["exact_match"]
f1_score = results["f1"]

print("\n" + "=" * 70)
print("SQuAD EVALUATION RESULTS")
print("=" * 70)

print(f"\nExact Match (EM) : {exact_match:.2f}")
print(f"F1 Score         : {f1_score:.2f}")

print("\n" + "=" * 70)
print("EVALUATION COMPLETE")
print("=" * 70)

# ---------------------------------------------------------------------
# 7. Store results for later README / analysis
# ---------------------------------------------------------------------

evaluation_results = {
    "exact_match": exact_match,
    "f1": f1_score,
    "validation_examples": len(references),
    "validation_features": len(validation_features),
    "training_time": "25:42",
    "validation_inference_time": "23.63 seconds",
    "answer_decoding_time": "25.71 seconds",
}

print("\nEvaluation results saved in `evaluation_results`.")

OFFICIAL SQuAD EVALUATION

[1] Loading SQuAD evaluation metric...


Metric loaded successfully.

[2] Building ground-truth references...
References : 10,570
Predictions : 10,570

[3] ID validation
Missing predictions : 0
Extra predictions   : 0
PASS — Prediction/reference IDs match.

[4] Computing EM and F1...

SQuAD EVALUATION RESULTS

Exact Match (EM) : 77.31
F1 Score         : 85.54

EVALUATION COMPLETE

Evaluation results saved in `evaluation_results`.


In [29]:
# Task 6 — Question Answering with Transformers
# Step 19: Qualitative Error Analysis

import re
import pandas as pd

print("=" * 70)
print("QUALITATIVE ERROR ANALYSIS")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Build analysis dataframe
# ---------------------------------------------------------------------

print("\n[1] Building analysis dataset...")

analysis_rows = []

for example in validation_examples:

    example_id = example["id"]

    prediction_row = predictions_df[
        predictions_df["id"] == example_id
    ]

    if prediction_row.empty:
        continue

    prediction = prediction_row.iloc[0]["prediction_text"]

    reference_answers = example["answers"]["text"]

    # SQuAD examples may contain multiple reference answers.
    reference = reference_answers[0]

    analysis_rows.append(
        {
            "id": example_id,
            "question": example["question"],
            "context": example["context"],
            "reference": reference,
            "prediction": prediction,
        }
    )

analysis_df = pd.DataFrame(analysis_rows)

print(f"Analysis examples : {len(analysis_df):,}")

# ---------------------------------------------------------------------
# 2. SQuAD-style normalization
# ---------------------------------------------------------------------

def normalize_answer(text):
    """Normalize text using the standard SQuAD-style rules."""

    text = text.lower()

    text = re.sub(
        r"\b(a|an|the)\b",
        " ",
        text,
    )

    text = re.sub(
        r"[^\w\s]",
        "",
        text,
    )

    text = " ".join(text.split())

    return text


# ---------------------------------------------------------------------
# 3. Token-level F1 helper
# ---------------------------------------------------------------------

def token_f1(prediction, reference):

    prediction_tokens = normalize_answer(
        prediction
    ).split()

    reference_tokens = normalize_answer(
        reference
    ).split()

    if not prediction_tokens or not reference_tokens:
        return 0.0

    common_tokens = set(prediction_tokens) & set(
        reference_tokens
    )

    num_common = sum(
        min(
            prediction_tokens.count(token),
            reference_tokens.count(token),
        )
        for token in common_tokens
    )

    if num_common == 0:
        return 0.0

    precision = (
        num_common / len(prediction_tokens)
    )

    recall = (
        num_common / len(reference_tokens)
    )

    return (
        2 * precision * recall
        / (precision + recall)
    )


# ---------------------------------------------------------------------
# 4. Calculate qualitative metrics
# ---------------------------------------------------------------------

print("\n[2] Calculating prediction-level analysis...")

analysis_df["normalized_prediction"] = (
    analysis_df["prediction"]
    .apply(normalize_answer)
)

analysis_df["normalized_reference"] = (
    analysis_df["reference"]
    .apply(normalize_answer)
)

analysis_df["exact_match"] = (
    analysis_df["normalized_prediction"]
    == analysis_df["normalized_reference"]
)

analysis_df["f1"] = analysis_df.apply(
    lambda row: token_f1(
        row["prediction"],
        row["reference"],
    ),
    axis=1,
)

analysis_df["prediction_length"] = (
    analysis_df["prediction"]
    .str.split()
    .str.len()
)

analysis_df["reference_length"] = (
    analysis_df["reference"]
    .str.split()
    .str.len()
)

# ---------------------------------------------------------------------
# 5. Summary
# ---------------------------------------------------------------------

print("\n[3] Analysis summary")

exact_match_rate = (
    analysis_df["exact_match"].mean() * 100
)

f1_mean = (
    analysis_df["f1"].mean() * 100
)

print(
    f"Exact Match : {exact_match_rate:.2f}%"
)

print(
    f"Mean F1     : {f1_mean:.2f}%"
)

print(
    f"Correct EM  : "
    f"{analysis_df['exact_match'].sum():,}"
)

print(
    f"Non-EM      : "
    f"{(~analysis_df['exact_match']).sum():,}"
)

# ---------------------------------------------------------------------
# 6. Perfect predictions
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE CORRECT PREDICTIONS")
print("=" * 70)

correct_df = analysis_df[
    analysis_df["exact_match"]
].copy()

display(
    correct_df[
        [
            "question",
            "reference",
            "prediction",
            "f1",
        ]
    ].head(10)
)

# ---------------------------------------------------------------------
# 7. Partial matches
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE PARTIAL MATCHES")
print("=" * 70)

partial_df = analysis_df[
    (~analysis_df["exact_match"])
    & (analysis_df["f1"] > 0)
    & (analysis_df["f1"] < 1)
].sort_values(
    "f1",
    ascending=False,
)

display(
    partial_df[
        [
            "question",
            "reference",
            "prediction",
            "f1",
        ]
    ].head(10)
)

# ---------------------------------------------------------------------
# 8. Completely wrong predictions
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("SAMPLE WRONG PREDICTIONS")
print("=" * 70)

wrong_df = analysis_df[
    analysis_df["f1"] == 0
].copy()

display(
    wrong_df[
        [
            "question",
            "reference",
            "prediction",
            "f1",
        ]
    ].head(10)
)

# ---------------------------------------------------------------------
# 9. Prediction length analysis
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("ANSWER LENGTH ANALYSIS")
print("=" * 70)

print(
    f"Average reference length  : "
    f"{analysis_df['reference_length'].mean():.2f} words"
)

print(
    f"Average prediction length : "
    f"{analysis_df['prediction_length'].mean():.2f} words"
)

# ---------------------------------------------------------------------
# 10. Final validation
# ---------------------------------------------------------------------

print("\n" + "=" * 70)
print("ERROR ANALYSIS COMPLETE")
print("=" * 70)

print(
    "\nAnalysis dataframe saved as `analysis_df`."
)

QUALITATIVE ERROR ANALYSIS

[1] Building analysis dataset...
Analysis examples : 10,570

[2] Calculating prediction-level analysis...

[3] Analysis summary
Exact Match : 64.04%
Mean F1     : 78.48%
Correct EM  : 6,769
Non-EM      : 3,801

SAMPLE CORRECT PREDICTIONS


,question,reference,prediction,f1
0,Which NFL team represented the AFC at Super Bo...,Denver Broncos,Denver Broncos,1.0
1,Which NFL team represented the NFC at Super Bo...,Carolina Panthers,Carolina Panthers,1.0
3,Which NFL team won Super Bowl 50?,Denver Broncos,Denver Broncos,1.0
4,What color was used to emphasize the 50th anni...,gold,gold,1.0
5,What was the theme of Super Bowl 50?,"""golden anniversary""","""golden anniversary""",1.0
6,What day was the game played on?,"February 7, 2016","February 7, 2016",1.0
7,What is the AFC short for?,American Football Conference,American Football Conference,1.0
8,What was the theme of Super Bowl 50?,"""golden anniversary""","""golden anniversary""",1.0
9,What does AFC stand for?,American Football Conference,American Football Conference,1.0
10,What day was the Super Bowl played on?,"February 7, 2016","February 7, 2016",1.0



SAMPLE PARTIAL MATCHES


,question,reference,prediction,f1
4325,Did the rainforest managed to thrive during th...,"However, the rainforest still managed to thriv...",the rainforest still managed to thrive during ...,0.974359
6429,What conditions must be met to prescribe a con...,it must be issued for a legitimate medical pur...,must be issued for a legitimate medical purpos...,0.971429
6427,What conditions must be met for a prescription...,it must be issued for a legitimate medical pur...,must be issued for a legitimate medical purpos...,0.971429
4859,WHat were features of Apple Talk,that allowed local area networks to be establi...,allowed local area networks to be established ...,0.969697
4873,What did Warner Sinback decide,"that a time-sharing system, based on Kemney's ...","a time-sharing system, based on Kemney's work ...",0.969697
8906,Why do land plants have more and smaller chlor...,to distribute chloroplasts so that they can ta...,ability to distribute chloroplasts so that the...,0.967742
4976,What did Shrewsbury note about the plague?,rates of mortality in rural areas during the 1...,the reported rates of mortality in rural areas...,0.967742
6461,Where might the doctor's self-interest be at o...,in obtaining cost-effective medication and avo...,obtaining cost-effective medication and avoidi...,0.962963
8948,What are we unsure of about how chloroplasts m...,whether the organelle carries out the last leg...,the organelle carries out the last leg of the ...,0.962963
4298,Why was the Dutch lawyer who moved to Belgium ...,because Dutch law said only people established...,Dutch law said only people established in the ...,0.960000



SAMPLE WRONG PREDICTIONS


,question,reference,prediction,f1
2,Where did Super Bowl 50 take place?,"Santa Clara, California",Levi's Stadium in the San Francisco Bay Area,0.0
16,What year did the Denver Broncos secure a Supe...,2015,2016,0.0
31,How many appearances have the Denver Broncos m...,8,eight,0.0
36,Who lost to the Broncos in the AFC Championship?,New England Patriots,Arizona Cardinals,0.0
37,Who were the defending Super Bowl champions?,New England Patriots,Arizona Cardinals,0.0
43,How many teams have played in the Super Bowl e...,4,four,0.0
47,How many times have the Panthers been in the S...,2,15–1,0.0
48,Who did Denver beat in the AFC championship?,New England Patriots,Arizona Cardinals,0.0
50,Who did Denver beat in the 2015 AFC Championsh...,New England Patriots,Arizona Cardinals,0.0
56,How many fumbles did Von Miller force in Super...,2,two,0.0



ANSWER LENGTH ANALYSIS
Average reference length  : 3.02 words
Average prediction length : 3.07 words

ERROR ANALYSIS COMPLETE

Analysis dataframe saved as `analysis_df`.


In [32]:
# Corrected Custom Extractive QA Inference Function
# Fix: pad overflow features before converting to PyTorch tensors

def answer_question(
    question,
    context,
    max_length=INFERENCE_MAX_LENGTH,
    doc_stride=INFERENCE_DOC_STRIDE,
    n_best=INFERENCE_N_BEST,
    max_answer_length=INFERENCE_MAX_ANSWER_LENGTH,
):
    """
    Extract an answer span from a context using the fine-tuned
    DistilBERT Question Answering model.

    Supports long contexts by using overflow windows with stride
    and dynamic padding.
    """

    if not isinstance(question, str) or not question.strip():
        raise ValueError(
            "Question must be a non-empty string."
        )

    if not isinstance(context, str) or not context.strip():
        raise ValueError(
            "Context must be a non-empty string."
        )

    question = question.strip()
    context = context.strip()

    # ---------------------------------------------------------------
    # Tokenization
    # ---------------------------------------------------------------

    encoded = inference_tokenizer(
        question,
        context,
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=True,
        return_tensors="pt",
    )

    # ---------------------------------------------------------------
    # Save metadata before removing it from model inputs
    # ---------------------------------------------------------------

    offset_mapping = encoded.pop(
        "offset_mapping"
    )

    encoded.pop(
        "overflow_to_sample_mapping"
    )

    # ---------------------------------------------------------------
    # Get sequence IDs for each overflow feature
    # ---------------------------------------------------------------

    sequence_ids = [
        encoded.sequence_ids(i)
        for i in range(
            encoded["input_ids"].shape[0]
        )
    ]

    # ---------------------------------------------------------------
    # Move model inputs to GPU
    # ---------------------------------------------------------------

    device = next(
        inference_model.parameters()
    ).device

    model_inputs = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    # ---------------------------------------------------------------
    # Model inference
    # ---------------------------------------------------------------

    with torch.no_grad():

        outputs = inference_model(
            **model_inputs
        )

    start_logits = (
        outputs.start_logits
        .detach()
        .cpu()
        .numpy()
    )

    end_logits = (
        outputs.end_logits
        .detach()
        .cpu()
        .numpy()
    )

    # ---------------------------------------------------------------
    # Generate answer candidates
    # ---------------------------------------------------------------

    candidates = []

    for feature_index in range(
        len(start_logits)
    ):

        feature_offsets = offset_mapping[
            feature_index
        ]

        feature_sequence_ids = sequence_ids[
            feature_index
        ]

        start_indexes = np.argsort(
            start_logits[feature_index]
        )[-n_best:][::-1]

        end_indexes = np.argsort(
            end_logits[feature_index]
        )[-n_best:][::-1]

        for start_index in start_indexes:

            # Start must belong to context.
            if (
                feature_sequence_ids[start_index]
                != 1
            ):
                continue

            start_offset = feature_offsets[
                start_index
            ]

            if start_offset is None:
                continue

            start_char = start_offset[0]

            for end_index in end_indexes:

                # End must belong to context.
                if (
                    feature_sequence_ids[end_index]
                    != 1
                ):
                    continue

                # Start must be before or equal to end.
                if start_index > end_index:
                    continue

                # Prevent excessively long answers.
                if (
                    end_index
                    - start_index
                    + 1
                    > max_answer_length
                ):
                    continue

                end_offset = feature_offsets[
                    end_index
                ]

                if end_offset is None:
                    continue

                end_char = end_offset[1]

                if start_char >= end_char:
                    continue

                if start_char < 0:
                    continue

                if end_char > len(context):
                    continue

                answer_text = context[
                    start_char:end_char
                ].strip()

                if not answer_text:
                    continue

                score = (
                    float(
                        start_logits[
                            feature_index,
                            start_index,
                        ]
                    )
                    +
                    float(
                        end_logits[
                            feature_index,
                            end_index,
                        ]
                    )
                )

                candidates.append(
                    {
                        "answer": answer_text,
                        "score": score,
                        "feature_index": feature_index,
                        "start_token": int(
                            start_index
                        ),
                        "end_token": int(
                            end_index
                        ),
                        "start_char": int(
                            start_char
                        ),
                        "end_char": int(
                            end_char
                        ),
                    }
                )

    # ---------------------------------------------------------------
    # No valid candidate
    # ---------------------------------------------------------------

    if not candidates:

        return {
            "answer": "",
            "score": float("-inf"),
            "feature_index": None,
            "start_token": None,
            "end_token": None,
            "start_char": None,
            "end_char": None,
            "num_features": int(
                encoded["input_ids"].shape[0]
            ),
            "num_candidates": 0,
        }

    # ---------------------------------------------------------------
    # Select best candidate
    # ---------------------------------------------------------------

    candidates.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    best = candidates[0]

    best["num_features"] = int(
        encoded["input_ids"].shape[0]
    )

    best["num_candidates"] = len(
        candidates
    )

    return best


print("Corrected inference function loaded successfully.")

Corrected inference function loaded successfully.


In [33]:
# Task 6 — Question Answering with Transformers
# Step 21: Long-Context Inference Validation

print("=" * 70)
print("LONG-CONTEXT INFERENCE VALIDATION")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Build a context longer than the model's maximum input length
# ---------------------------------------------------------------------

base_context = (
    "The University of Notre Dame is located "
    "in South Bend, Indiana. "
)

long_context = base_context * 90

# Put the target information near the end of the long context.
long_context += (
    "The university was founded in 1842 "
    "by Edward Sorin."
)

long_question = (
    "Who founded the University of Notre Dame?"
)

print("\n[1] Long-context setup")

print(
    f"Context characters : "
    f"{len(long_context):,}"
)

print(
    f"Context words     : "
    f"{len(long_context.split()):,}"
)

print(
    f"Question           : "
    f"{long_question}"
)

# ---------------------------------------------------------------------
# 2. Run custom inference
# ---------------------------------------------------------------------

print("\n[2] Running long-context inference...")

long_result = answer_question(
    question=long_question,
    context=long_context,
)

# ---------------------------------------------------------------------
# 3. Display result
# ---------------------------------------------------------------------

print("\n[3] Inference result")

print(
    f"Predicted answer   : "
    f"{long_result['answer']}"
)

print(
    f"Score              : "
    f"{long_result['score']:.4f}"
)

print(
    f"Features used      : "
    f"{long_result['num_features']}"
)

print(
    f"Candidates generated : "
    f"{long_result['num_candidates']}"
)

print(
    f"Character span     : "
    f"{long_result['start_char']}:"
    f"{long_result['end_char']}"
)

# ---------------------------------------------------------------------
# 4. Verify the answer came from the long context
# ---------------------------------------------------------------------

print("\n[4] Long-context validation")

expected_answer = "Edward Sorin"

if long_result["answer"].lower() == expected_answer.lower():

    print(
        "PASS — The model correctly extracted "
        "the answer from the long context."
    )

else:

    print(
        "WARNING — The prediction differs from "
        f"the expected answer: {expected_answer}"
    )

# ---------------------------------------------------------------------
# 5. Verify multiple features were actually created
# ---------------------------------------------------------------------

if long_result["num_features"] > 1:

    print(
        "PASS — Long context was split into "
        f"{long_result['num_features']} overflow features."
    )

else:

    print(
        "WARNING — Only one feature was created. "
        "The context may not be long enough to test overflow."
    )

print("\n" + "=" * 70)
print("LONG-CONTEXT VALIDATION COMPLETE")
print("=" * 70)

LONG-CONTEXT INFERENCE VALIDATION

[1] Long-context setup
Context characters : 5,811
Context words     : 999
Question           : Who founded the University of Notre Dame?

[2] Running long-context inference...

[3] Inference result
Predicted answer   : The University of Notre Dame is located in South Bend, Indiana
Score              : -14.3359
Features used      : 2
Candidates generated : 158
Character span     : 1408:1470

[4] Long-context validation
WARNING — The prediction differs from the expected answer: Edward Sorin
PASS — Long context was split into 2 overflow features.

LONG-CONTEXT VALIDATION COMPLETE


In [34]:
# Task 6 — Question Answering with Transformers
# Step 21B: Debug Long-Context Candidate Ranking

print("=" * 70)
print("LONG-CONTEXT CANDIDATE DEBUGGING")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Tokenize the same long context
# ---------------------------------------------------------------------

debug_encoded = inference_tokenizer(
    long_question,
    long_context,
    truncation="only_second",
    max_length=INFERENCE_MAX_LENGTH,
    stride=INFERENCE_DOC_STRIDE,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding=True,
    return_tensors="pt",
)

debug_offsets = debug_encoded.pop(
    "offset_mapping"
)

debug_encoded.pop(
    "overflow_to_sample_mapping"
)

debug_sequence_ids = [
    debug_encoded.sequence_ids(i)
    for i in range(
        debug_encoded["input_ids"].shape[0]
    )
]

debug_device = next(
    inference_model.parameters()
).device

debug_inputs = {
    key: value.to(debug_device)
    for key, value in debug_encoded.items()
}

# ---------------------------------------------------------------------
# 2. Run model
# ---------------------------------------------------------------------

with torch.no_grad():

    debug_outputs = inference_model(
        **debug_inputs
    )

debug_start_logits = (
    debug_outputs.start_logits
    .detach()
    .cpu()
    .numpy()
)

debug_end_logits = (
    debug_outputs.end_logits
    .detach()
    .cpu()
    .numpy()
)

# ---------------------------------------------------------------------
# 3. Locate expected answer in original context
# ---------------------------------------------------------------------

expected_answer = "Edward Sorin"

expected_start = long_context.find(
    expected_answer
)

expected_end = (
    expected_start
    + len(expected_answer)
)

print("\n[1] Expected answer location")

print(
    f"Expected answer : {expected_answer}"
)

print(
    f"Character span  : "
    f"{expected_start}:{expected_end}"
)

# ---------------------------------------------------------------------
# 4. Find which feature contains expected answer
# ---------------------------------------------------------------------

print("\n[2] Feature coverage")

expected_features = []

for feature_index in range(
    len(debug_start_logits)
):

    offsets = debug_offsets[
        feature_index
    ]

    context_token_indices = [
        index
        for index, sequence_id
        in enumerate(
            debug_sequence_ids[
                feature_index
            ]
        )
        if sequence_id == 1
        and offsets[index] is not None
    ]

    if not context_token_indices:
        continue

    feature_start_char = min(
        offsets[index][0]
        for index in context_token_indices
    )

    feature_end_char = max(
        offsets[index][1]
        for index in context_token_indices
    )

    print(
        f"Feature {feature_index}: "
        f"context chars "
        f"{feature_start_char}:{feature_end_char}"
    )

    if (
        expected_start >= feature_start_char
        and expected_end <= feature_end_char
    ):
        expected_features.append(
            feature_index
        )

print(
    f"\nFeatures containing expected answer: "
    f"{expected_features}"
)

# ---------------------------------------------------------------------
# 5. Inspect logits around expected answer
# ---------------------------------------------------------------------

print("\n[3] Expected-answer token scores")

for feature_index in expected_features:

    offsets = debug_offsets[
        feature_index
    ]

    matching_tokens = []

    for token_index, offset in enumerate(
        offsets
    ):

        if offset is None:
            continue

        if (
            offset[0] >= expected_start
            and offset[1] <= expected_end
        ):

            matching_tokens.append(
                token_index
            )

    print(
        f"\nFeature {feature_index}"
    )

    print(
        f"Matching token indices: "
        f"{matching_tokens}"
    )

    for token_index in matching_tokens:

        start_score = float(
            debug_start_logits[
                feature_index,
                token_index,
            ]
        )

        end_score = float(
            debug_end_logits[
                feature_index,
                token_index,
            ]
        )

        print(
            f"Token {token_index:3d} | "
            f"offset={offsets[token_index]} | "
            f"start={start_score:8.4f} | "
            f"end={end_score:8.4f}"
        )

# ---------------------------------------------------------------------
# 6. Show top start and end tokens in each feature
# ---------------------------------------------------------------------

print("\n[4] Top context start/end logits")

for feature_index in range(
    len(debug_start_logits)
):

    valid_context_indices = [
        index
        for index, sequence_id
        in enumerate(
            debug_sequence_ids[
                feature_index
            ]
        )
        if sequence_id == 1
        and debug_offsets[
            feature_index
        ][index] is not None
    ]

    top_starts = sorted(
        valid_context_indices,
        key=lambda index:
            debug_start_logits[
                feature_index,
                index,
            ],
        reverse=True,
    )[:5]

    top_ends = sorted(
        valid_context_indices,
        key=lambda index:
            debug_end_logits[
                feature_index,
                index,
            ],
        reverse=True,
    )[:5]

    print(
        f"\nFeature {feature_index}"
    )

    print("Top start tokens:")

    for index in top_starts:

        offset = debug_offsets[
            feature_index
        ][index]

        text = long_context[
            offset[0]:offset[1]
        ]

        print(
            f"  token={index:3d} "
            f"score={debug_start_logits[feature_index, index]:8.4f} "
            f"text='{text}'"
        )

    print("Top end tokens:")

    for index in top_ends:

        offset = debug_offsets[
            feature_index
        ][index]

        text = long_context[
            offset[0]:offset[1]
        ]

        print(
            f"  token={index:3d} "
            f"score={debug_end_logits[feature_index, index]:8.4f} "
            f"text='{text}'"
        )

print("\n" + "=" * 70)
print("LONG-CONTEXT DEBUGGING COMPLETE")
print("=" * 70)

LONG-CONTEXT CANDIDATE DEBUGGING

[1] Expected answer location
Expected answer : Edward Sorin
Character span  : 5798:5810

[2] Feature coverage
Feature 0: context chars 0:1840
Feature 1: context chars 1207:1895

Features containing expected answer: []

[3] Expected-answer token scores

[4] Top context start/end logits

Feature 0
Top start tokens:
  token= 36 score= -7.6445 text='The'
  token= 49 score= -7.6914 text='The'
  token= 23 score= -7.8164 text='The'
  token= 62 score= -7.8242 text='The'
  token= 75 score= -7.8828 text='The'
Top end tokens:
  token= 60 score= -7.8633 text='Indiana'
  token= 86 score= -7.9453 text='Indiana'
  token= 47 score= -8.0156 text='Indiana'
  token= 34 score= -8.0703 text='Indiana'
  token= 73 score= -8.0703 text='Indiana'

Feature 1
Top start tokens:
  token= 51 score= -6.9102 text='The'
  token= 38 score= -6.9141 text='The'
  token= 64 score= -7.0000 text='The'
  token= 25 score= -7.1445 text='The'
  token= 77 score= -7.2656 text='The'
Top end tokens:


In [35]:
# Task 6 — Question Answering with Transformers
# Step 21C: Validate Tokenizer Overflow Windows

print("=" * 70)
print("TOKENIZER OVERFLOW VALIDATION")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Tokenize long context WITHOUT converting to tensors
# ---------------------------------------------------------------------

overflow_test = inference_tokenizer(
    long_question,
    long_context,
    truncation="only_second",
    max_length=INFERENCE_MAX_LENGTH,
    stride=INFERENCE_DOC_STRIDE,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding=False,
)

# ---------------------------------------------------------------------
# 2. Inspect number of features
# ---------------------------------------------------------------------

num_features = len(
    overflow_test["input_ids"]
)

print("\n[1] Overflow features")

print(
    f"Number of features : "
    f"{num_features}"
)

print(
    f"Expected: more than 2 for a "
    f"{len(long_context.split()):,}-word context"
)

# ---------------------------------------------------------------------
# 3. Inspect feature lengths
# ---------------------------------------------------------------------

feature_lengths = [
    len(input_ids)
    for input_ids in overflow_test["input_ids"]
]

print("\n[2] Feature lengths")

print(
    f"Minimum : {min(feature_lengths)}"
)

print(
    f"Maximum : {max(feature_lengths)}"
)

print(
    f"Average : "
    f"{sum(feature_lengths) / len(feature_lengths):.2f}"
)

# ---------------------------------------------------------------------
# 4. Find expected answer coverage
# ---------------------------------------------------------------------

expected_answer = "Edward Sorin"

expected_start = long_context.find(
    expected_answer
)

expected_end = (
    expected_start
    + len(expected_answer)
)

print("\n[3] Expected answer")

print(
    f"Answer : {expected_answer}"
)

print(
    f"Span   : "
    f"{expected_start}:{expected_end}"
)

answer_features = []

for feature_index, offsets in enumerate(
    overflow_test["offset_mapping"]
):

    context_offsets = [
        offset
        for offset in offsets
        if offset is not None
    ]

    if not context_offsets:
        continue

    feature_start = min(
        offset[0]
        for offset in context_offsets
    )

    feature_end = max(
        offset[1]
        for offset in context_offsets
    )

    if (
        expected_start >= feature_start
        and expected_end <= feature_end
    ):
        answer_features.append(
            feature_index
        )

print(
    f"Features containing answer: "
    f"{answer_features}"
)

# ---------------------------------------------------------------------
# 5. Print feature coverage
# ---------------------------------------------------------------------

print("\n[4] Feature character coverage")

for feature_index, offsets in enumerate(
    overflow_test["offset_mapping"]
):

    context_offsets = [
        offset
        for offset in offsets
        if offset is not None
    ]

    if not context_offsets:
        continue

    feature_start = min(
        offset[0]
        for offset in context_offsets
    )

    feature_end = max(
        offset[1]
        for offset in context_offsets
    )

    print(
        f"Feature {feature_index:2d}: "
        f"{feature_start}:{feature_end}"
    )

# ---------------------------------------------------------------------
# 6. Final validation
# ---------------------------------------------------------------------

print("\n[5] Validation")

if num_features > 2:
    print(
        "PASS — Multiple overflow windows "
        "were generated."
    )
else:
    print(
        "WARNING — Unexpectedly few overflow "
        "windows were generated."
    )

if answer_features:
    print(
        "PASS — Expected answer is covered "
        "by at least one overflow window."
    )
else:
    print(
        "FAIL — Expected answer is not covered "
        "by any overflow window."
    )

print("\n" + "=" * 70)
print("TOKENIZER OVERFLOW VALIDATION COMPLETE")
print("=" * 70)

TOKENIZER OVERFLOW VALIDATION

[1] Overflow features
Number of features : 2
Expected: more than 2 for a 999-word context

[2] Feature lengths
Minimum : 150
Maximum : 384
Average : 267.00

[3] Expected answer
Answer : Edward Sorin
Span   : 5798:5810
Features containing answer: []

[4] Feature character coverage
Feature  0: 0:1840
Feature  1: 0:1895

[5] Validation
WARNING — Unexpectedly few overflow windows were generated.
FAIL — Expected answer is not covered by any overflow window.

TOKENIZER OVERFLOW VALIDATION COMPLETE


In [36]:
# Task 6 — Question Answering with Transformers
# Step 21D: Correct Long-Context Test

print("=" * 70)
print("CORRECT LONG-CONTEXT TEST")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Build a realistic long context
# ---------------------------------------------------------------------

filler_sentence = (
    "The University of Notre Dame is a private "
    "research university located in Indiana. "
)

long_context = " ".join(
    [filler_sentence] * 120
)

long_context += (
    "The university was founded in 1842 "
    "by Edward Sorin."
)

long_question = (
    "Who founded the University of Notre Dame?"
)

print("\n[1] Context")

print(
    f"Characters : {len(long_context):,}"
)

print(
    f"Words      : {len(long_context.split()):,}"
)

print(
    f"Question   : {long_question}"
)

# ---------------------------------------------------------------------
# 2. Inspect tokenizer overflow
# ---------------------------------------------------------------------

overflow_test = inference_tokenizer(
    long_question,
    long_context,
    truncation="only_second",
    max_length=INFERENCE_MAX_LENGTH,
    stride=INFERENCE_DOC_STRIDE,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding=False,
)

num_features = len(
    overflow_test["input_ids"]
)

print("\n[2] Tokenizer overflow")

print(
    f"Number of features : "
    f"{num_features}"
)

feature_lengths = [
    len(ids)
    for ids in overflow_test["input_ids"]
]

print(
    f"Feature lengths    : "
    f"min={min(feature_lengths)}, "
    f"max={max(feature_lengths)}"
)

# ---------------------------------------------------------------------
# 3. Locate expected answer
# ---------------------------------------------------------------------

expected_answer = "Edward Sorin"

expected_start = long_context.find(
    expected_answer
)

expected_end = (
    expected_start
    + len(expected_answer)
)

print("\n[3] Expected answer")

print(
    f"Answer : {expected_answer}"
)

print(
    f"Span   : "
    f"{expected_start}:{expected_end}"
)

# ---------------------------------------------------------------------
# 4. Check which overflow feature contains answer
# ---------------------------------------------------------------------

answer_features = []

for feature_index, offsets in enumerate(
    overflow_test["offset_mapping"]
):

    context_offsets = [
        offset
        for offset in offsets
        if offset is not None
    ]

    if not context_offsets:
        continue

    feature_start = min(
        offset[0]
        for offset in context_offsets
    )

    feature_end = max(
        offset[1]
        for offset in context_offsets
    )

    if (
        expected_start >= feature_start
        and expected_end <= feature_end
    ):

        answer_features.append(
            feature_index
        )

    print(
        f"Feature {feature_index:2d}: "
        f"{feature_start}:{feature_end}"
    )

print(
    f"\nAnswer-containing features: "
    f"{answer_features}"
)

# ---------------------------------------------------------------------
# 5. Run actual custom inference
# ---------------------------------------------------------------------

print("\n[4] Running custom inference...")

long_result = answer_question(
    question=long_question,
    context=long_context,
)

print(
    f"\nPredicted answer : "
    f"{long_result['answer']}"
)

print(
    f"Score            : "
    f"{long_result['score']:.4f}"
)

print(
    f"Features used    : "
    f"{long_result['num_features']}"
)

print(
    f"Candidates       : "
    f"{long_result['num_candidates']}"
)

print(
    f"Character span   : "
    f"{long_result['start_char']}:"
    f"{long_result['end_char']}"
)

# ---------------------------------------------------------------------
# 6. Final validation
# ---------------------------------------------------------------------

print("\n[5] Validation")

if num_features > 1:
    print(
        "PASS — Multiple overflow features generated."
    )
else:
    print(
        "FAIL — Long context did not create "
        "multiple features."
    )

if answer_features:
    print(
        "PASS — Expected answer is covered "
        "by an overflow feature."
    )
else:
    print(
        "FAIL — Expected answer is not covered "
        "by any feature."
    )

if (
    long_result["answer"].lower()
    == expected_answer.lower()
):

    print(
        "PASS — Custom inference extracted "
        "the expected answer."
    )

else:

    print(
        "WARNING — Custom inference prediction "
        "differs from the expected answer."
    )

print("\n" + "=" * 70)
print("CORRECT LONG-CONTEXT TEST COMPLETE")
print("=" * 70)

CORRECT LONG-CONTEXT TEST

[1] Context
Characters : 10,010
Words      : 1,569
Question   : Who founded the University of Notre Dame?

[2] Tokenizer overflow
Number of features : 2
Feature lengths    : min=150, max=384

[3] Expected answer
Answer : Edward Sorin
Span   : 9997:10009
Feature  0: 0:2208
Feature  1: 0:2272

Answer-containing features: []

[4] Running custom inference...

Predicted answer : The University of Notre Dame is a private research university located in Indiana
Score            : -13.3594
Features used    : 2
Candidates       : 108
Character span   : 1494:1574

[5] Validation
PASS — Multiple overflow features generated.
FAIL — Expected answer is not covered by any feature.
WARNING — Custom inference prediction differs from the expected answer.

CORRECT LONG-CONTEXT TEST COMPLETE


In [37]:
# Task 6 — Question Answering with Transformers
# Step 21E: Diagnose Fast Tokenizer Overflow Mapping

print("=" * 70)
print("FAST TOKENIZER OVERFLOW DIAGNOSTIC")
print("=" * 70)

diagnostic = inference_tokenizer(
    long_question,
    long_context,
    truncation="only_second",
    max_length=INFERENCE_MAX_LENGTH,
    stride=INFERENCE_DOC_STRIDE,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding=False,
)

print("\n[1] Basic output")

print(
    "Number of features:",
    len(diagnostic["input_ids"])
)

print(
    "overflow_to_sample_mapping:",
    diagnostic.get("overflow_to_sample_mapping")
)

# ---------------------------------------------------------------------
# Inspect every feature
# ---------------------------------------------------------------------

print("\n[2] Feature diagnostics")

for i in range(len(diagnostic["input_ids"])):

    input_ids = diagnostic["input_ids"][i]
    offsets = diagnostic["offset_mapping"][i]

    sequence_ids = diagnostic.sequence_ids(i)

    context_offsets = []

    for offset, sequence_id in zip(
        offsets,
        sequence_ids
    ):

        if (
            sequence_id == 1
            and offset[1] > offset[0]
        ):
            context_offsets.append(offset)

    if context_offsets:

        min_char = min(
            offset[0]
            for offset in context_offsets
        )

        max_char = max(
            offset[1]
            for offset in context_offsets
        )

    else:

        min_char = None
        max_char = None

    print(
        f"\nFeature {i}"
    )

    print(
        f"  Tokens          : {len(input_ids)}"
    )

    print(
        f"  Context chars   : "
        f"{min_char}:{max_char}"
    )

    print(
        f"  First 10 IDs    : "
        f"{input_ids[:10]}"
    )

    print(
        f"  Last 10 IDs     : "
        f"{input_ids[-10:]}"
    )

# ---------------------------------------------------------------------
# Compare with a single-context tokenizer call
# ---------------------------------------------------------------------

print("\n[3] Single sequence diagnostic")

single = inference_tokenizer(
    long_context,
    truncation=True,
    max_length=INFERENCE_MAX_LENGTH,
    stride=INFERENCE_DOC_STRIDE,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding=False,
)

print(
    "Single-sequence features:",
    len(single["input_ids"])
)

for i, offsets in enumerate(
    single["offset_mapping"]
):

    valid_offsets = [
        offset
        for offset in offsets
        if offset[1] > offset[0]
    ]

    if valid_offsets:

        min_char = min(
            offset[0]
            for offset in valid_offsets
        )

        max_char = max(
            offset[1]
            for offset in valid_offsets
        )

    else:

        min_char = None
        max_char = None

    print(
        f"  Feature {i}: "
        f"{min_char}:{max_char}, "
        f"tokens={len(single['input_ids'][i])}"
    )

print("\n" + "=" * 70)
print("FAST TOKENIZER OVERFLOW DIAGNOSTIC COMPLETE")
print("=" * 70)

FAST TOKENIZER OVERFLOW DIAGNOSTIC

[1] Basic output
Number of features: 2
overflow_to_sample_mapping: [0, 0]

[2] Feature diagnostics

Feature 0
  Tokens          : 384
  Context chars   : 0:2208
  First 10 IDs    : [101, 2040, 2631, 1996, 2118, 1997, 10289, 8214, 1029, 102]
  Last 10 IDs     : [1996, 2118, 1997, 10289, 8214, 2003, 1037, 2797, 2470, 102]

Feature 1
  Tokens          : 150
  Context chars   : 1445:2272
  First 10 IDs    : [101, 2040, 2631, 1996, 2118, 1997, 10289, 8214, 1029, 102]
  Last 10 IDs     : [1999, 5242, 1012, 1996, 2118, 1997, 10289, 8214, 2003, 102]

[3] Single sequence diagnostic
Single-sequence features: 2
  Feature 0: 0:2264, tokens=384
  Feature 1: 1509:2272, tokens=132

FAST TOKENIZER OVERFLOW DIAGNOSTIC COMPLETE


In [38]:
# Task 6 — Question Answering with Transformers
# Step 22: Robust Long-Context Inference

import time
import torch

INFERENCE_MAX_LENGTH = 384
INFERENCE_DOC_STRIDE = 128
INFERENCE_N_BEST = 20
INFERENCE_MAX_ANSWER_LENGTH = 30

def answer_question(
    question,
    context,
    max_length=INFERENCE_MAX_LENGTH,
    doc_stride=INFERENCE_DOC_STRIDE,
    n_best=INFERENCE_N_BEST,
    max_answer_length=INFERENCE_MAX_ANSWER_LENGTH,
):
    """
    Extractive QA inference with explicit sliding windows.

    The context is tokenized independently and split into overlapping
    windows. Each window is then combined with the question and passed
    through the fine-tuned DistilBERT QA model.

    Returns the highest-scoring valid answer span.
    """

    start_time = time.time()

    question = question.strip()
    context = context.strip()

    if not question:
        raise ValueError("Question cannot be empty.")

    if not context:
        raise ValueError("Context cannot be empty.")

    # ---------------------------------------------------------------
    # 1. Tokenize question
    # ---------------------------------------------------------------

    question_tokens = inference_tokenizer(
        question,
        add_special_tokens=False,
    )

    question_ids = question_tokens["input_ids"]

    # ---------------------------------------------------------------
    # 2. Tokenize context while preserving character offsets
    # ---------------------------------------------------------------

    context_tokens = inference_tokenizer(
        context,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    context_ids = context_tokens["input_ids"]
    context_offsets = context_tokens["offset_mapping"]

    # ---------------------------------------------------------------
    # 3. Calculate available context tokens per window
    # ---------------------------------------------------------------

    special_tokens_count = (
        inference_tokenizer.num_special_tokens_to_add(
            pair=True
        )
    )

    max_context_tokens = (
        max_length
        - len(question_ids)
        - special_tokens_count
    )

    if max_context_tokens <= 0:
        raise ValueError(
            "Question is too long for the selected max_length."
        )

    # ---------------------------------------------------------------
    # 4. Build overlapping context windows
    # ---------------------------------------------------------------

    step = max_context_tokens - doc_stride

    if step <= 0:
        raise ValueError(
            "doc_stride must be smaller than the available "
            "context window."
        )

    windows = []

    start_token = 0

    while start_token < len(context_ids):

        end_token = min(
            start_token + max_context_tokens,
            len(context_ids),
        )

        window_ids = context_ids[
            start_token:end_token
        ]

        window_offsets = context_offsets[
            start_token:end_token
        ]

        windows.append(
            {
                "context_ids": window_ids,
                "context_offsets": window_offsets,
                "context_start_token": start_token,
                "context_end_token": end_token,
            }
        )

        if end_token >= len(context_ids):
            break

        start_token += step

    # ---------------------------------------------------------------
    # 5. Build question + context feature for every window
    # ---------------------------------------------------------------

    features = []

    for window in windows:

        encoded = inference_tokenizer(
            question,
            context,
            add_special_tokens=True,
            truncation="only_second",
            max_length=max_length,
            padding=False,
        )

        # -----------------------------------------------------------
        # Replace tokenizer-selected context with our explicit window
        # -----------------------------------------------------------

        question_ids_with_special = (
            [inference_tokenizer.cls_token_id]
            + question_ids
            + [inference_tokenizer.sep_token_id]
        )

        window_input_ids = (
            question_ids_with_special
            + window["context_ids"]
            + [inference_tokenizer.sep_token_id]
        )

        # -----------------------------------------------------------
        # Construct attention mask
        # -----------------------------------------------------------

        attention_mask = [
            1
            for _ in window_input_ids
        ]

        # -----------------------------------------------------------
        # Context token positions inside the final sequence
        # -----------------------------------------------------------

        context_start_position = (
            len(question_ids_with_special)
        )

        context_end_position = (
            context_start_position
            + len(window["context_ids"])
            - 1
        )

        features.append(
            {
                "input_ids": window_input_ids,
                "attention_mask": attention_mask,
                "context_offsets": window[
                    "context_offsets"
                ],
                "context_start_position":
                    context_start_position,
                "context_end_position":
                    context_end_position,
            }
        )

    # ---------------------------------------------------------------
    # 6. Batch features dynamically
    # ---------------------------------------------------------------

    model_features = [
        {
            "input_ids": feature["input_ids"],
            "attention_mask": feature["attention_mask"],
        }
        for feature in features
    ]

    batch = data_collator(
        model_features
    )

    batch = {
        key: value.to(saved_model.device)
        for key, value in batch.items()
    }

    # ---------------------------------------------------------------
    # 7. Model inference
    # ---------------------------------------------------------------

    saved_model.eval()

    with torch.no_grad():

        outputs = saved_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    # ---------------------------------------------------------------
    # 8. Generate valid answer candidates
    # ---------------------------------------------------------------

    candidates = []

    for feature_index, feature in enumerate(features):

        start_scores = (
            start_logits[
                feature_index
            ].detach().float().cpu()
        )

        end_scores = (
            end_logits[
                feature_index
            ].detach().float().cpu()
        )

        context_start = (
            feature["context_start_position"]
        )

        context_end = (
            feature["context_end_position"]
        )

        valid_start_scores = (
            start_scores[
                context_start:
                context_end + 1
            ]
        )

        valid_end_scores = (
            end_scores[
                context_start:
                context_end + 1
            ]
        )

        top_start = torch.topk(
            valid_start_scores,
            k=min(
                n_best,
                len(valid_start_scores),
            ),
        )

        top_end = torch.topk(
            valid_end_scores,
            k=min(
                n_best,
                len(valid_end_scores),
            ),
        )

        offsets = feature[
            "context_offsets"
        ]

        for start_position, start_score in zip(
            top_start.indices.tolist(),
            top_start.values.tolist(),
        ):

            for end_position, end_score in zip(
                top_end.indices.tolist(),
                top_end.values.tolist(),
            ):

                if end_position < start_position:
                    continue

                span_length = (
                    end_position
                    - start_position
                    + 1
                )

                if span_length > max_answer_length:
                    continue

                if (
                    start_position >= len(offsets)
                    or end_position >= len(offsets)
                ):
                    continue

                char_start = offsets[
                    start_position
                ][0]

                char_end = offsets[
                    end_position
                ][1]

                if char_end <= char_start:
                    continue

                answer_text = context[
                    char_start:char_end
                ].strip()

                if not answer_text:
                    continue

                score = (
                    start_score
                    + end_score
                )

                candidates.append(
                    {
                        "answer": answer_text,
                        "score": score,
                        "start_char": char_start,
                        "end_char": char_end,
                        "feature_index":
                            feature_index,
                    }
                )

    # ---------------------------------------------------------------
    # 9. Select best candidate
    # ---------------------------------------------------------------

    if not candidates:

        elapsed = time.time() - start_time

        return {
            "answer": "",
            "score": float("-inf"),
            "start_char": None,
            "end_char": None,
            "feature_index": None,
            "num_features": len(features),
            "num_candidates": 0,
            "inference_time": elapsed,
        }

    candidates.sort(
        key=lambda x: x["score"],
        reverse=True,
    )

    best = candidates[0]

    elapsed = time.time() - start_time

    return {
        "answer": best["answer"],
        "score": best["score"],
        "start_char": best["start_char"],
        "end_char": best["end_char"],
        "feature_index": best["feature_index"],
        "num_features": len(features),
        "num_candidates": len(candidates),
        "inference_time": elapsed,
    }


print("=" * 70)
print("ROBUST LONG-CONTEXT INFERENCE FUNCTION")
print("=" * 70)

print(
    "MAX_LENGTH        :",
    INFERENCE_MAX_LENGTH
)

print(
    "DOC_STRIDE        :",
    INFERENCE_DOC_STRIDE
)

print(
    "N_BEST            :",
    INFERENCE_N_BEST
)

print(
    "MAX_ANSWER_LENGTH :",
    INFERENCE_MAX_ANSWER_LENGTH
)

print(
    "\nFunction created successfully."
)

print("=" * 70)

ROBUST LONG-CONTEXT INFERENCE FUNCTION
MAX_LENGTH        : 384
DOC_STRIDE        : 128
N_BEST            : 20
MAX_ANSWER_LENGTH : 30

Function created successfully.


In [40]:
# Task 6 — Question Answering with Transformers
# Step 24: Clean Production Inference + Real SQuAD Test

import time
import torch

def answer_question(
    question,
    context,
    max_length=INFERENCE_MAX_LENGTH,
    doc_stride=INFERENCE_DOC_STRIDE,
    n_best=INFERENCE_N_BEST,
    max_answer_length=INFERENCE_MAX_ANSWER_LENGTH,
):
    """
    Production-ready extractive QA inference.

    Uses explicit sliding windows over context tokens to support
    long contexts beyond the model's maximum sequence length.
    """

    start_time = time.time()

    question = question.strip()
    context = context.strip()

    if not question:
        raise ValueError("Question cannot be empty.")

    if not context:
        raise ValueError("Context cannot be empty.")

    # ---------------------------------------------------------------
    # 1. Tokenize question
    # ---------------------------------------------------------------

    question_encoding = inference_tokenizer(
        question,
        add_special_tokens=False,
    )

    question_ids = question_encoding["input_ids"]

    # ---------------------------------------------------------------
    # 2. Tokenize context only
    # ---------------------------------------------------------------

    context_encoding = inference_tokenizer(
        context,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    context_ids = context_encoding["input_ids"]

    context_offsets = context_encoding[
        "offset_mapping"
    ]

    # ---------------------------------------------------------------
    # 3. Calculate available context capacity
    # ---------------------------------------------------------------

    special_tokens_count = (
        inference_tokenizer.num_special_tokens_to_add(
            pair=True
        )
    )

    max_context_tokens = (
        max_length
        - len(question_ids)
        - special_tokens_count
    )

    if max_context_tokens <= 0:
        raise ValueError(
            "Question is too long for max_length."
        )

    step = max_context_tokens - doc_stride

    if step <= 0:
        raise ValueError(
            "doc_stride must be smaller than "
            "the available context window."
        )

    # ---------------------------------------------------------------
    # 4. Create explicit sliding windows
    # ---------------------------------------------------------------

    windows = []

    start_token = 0

    while start_token < len(context_ids):

        end_token = min(
            start_token + max_context_tokens,
            len(context_ids),
        )

        windows.append(
            {
                "context_ids":
                    context_ids[
                        start_token:end_token
                    ],

                "context_offsets":
                    context_offsets[
                        start_token:end_token
                    ],
            }
        )

        if end_token >= len(context_ids):
            break

        start_token += step

    # ---------------------------------------------------------------
    # 5. Build model features
    # ---------------------------------------------------------------

    features = []

    question_prefix = (
        [inference_tokenizer.cls_token_id]
        + question_ids
        + [inference_tokenizer.sep_token_id]
    )

    for window in windows:

        input_ids = (
            question_prefix
            + window["context_ids"]
            + [inference_tokenizer.sep_token_id]
        )

        attention_mask = [
            1
            for _ in input_ids
        ]

        context_start_position = len(
            question_prefix
        )

        context_end_position = (
            context_start_position
            + len(window["context_ids"])
            - 1
        )

        features.append(
            {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "context_offsets":
                    window["context_offsets"],
                "context_start_position":
                    context_start_position,
                "context_end_position":
                    context_end_position,
            }
        )

    # ---------------------------------------------------------------
    # 6. Dynamic padding
    # ---------------------------------------------------------------

    model_features = [
        {
            "input_ids": feature["input_ids"],
            "attention_mask":
                feature["attention_mask"],
        }
        for feature in features
    ]

    batch = data_collator(
        model_features
    )

    batch = {
        key: value.to(saved_model.device)
        for key, value in batch.items()
    }

    # ---------------------------------------------------------------
    # 7. Model inference
    # ---------------------------------------------------------------

    saved_model.eval()

    with torch.no_grad():

        outputs = saved_model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

    start_logits = (
        outputs.start_logits
        .detach()
        .float()
        .cpu()
    )

    end_logits = (
        outputs.end_logits
        .detach()
        .float()
        .cpu()
    )

    # ---------------------------------------------------------------
    # 8. Candidate generation
    # ---------------------------------------------------------------

    candidates = []

    for feature_index, feature in enumerate(
        features
    ):

        context_start = (
            feature["context_start_position"]
        )

        context_end = (
            feature["context_end_position"]
        )

        offsets = feature[
            "context_offsets"
        ]

        feature_start_logits = (
            start_logits[
                feature_index
            ]
        )

        feature_end_logits = (
            end_logits[
                feature_index
            ]
        )

        valid_start_logits = (
            feature_start_logits[
                context_start:
                context_end + 1
            ]
        )

        valid_end_logits = (
            feature_end_logits[
                context_start:
                context_end + 1
            ]
        )

        start_k = min(
            n_best,
            len(valid_start_logits),
        )

        end_k = min(
            n_best,
            len(valid_end_logits),
        )

        top_starts = torch.topk(
            valid_start_logits,
            k=start_k,
        )

        top_ends = torch.topk(
            valid_end_logits,
            k=end_k,
        )

        for start_position, start_score in zip(
            top_starts.indices.tolist(),
            top_starts.values.tolist(),
        ):

            for end_position, end_score in zip(
                top_ends.indices.tolist(),
                top_ends.values.tolist(),
            ):

                if end_position < start_position:
                    continue

                span_length = (
                    end_position
                    - start_position
                    + 1
                )

                if (
                    span_length
                    > max_answer_length
                ):
                    continue

                if (
                    start_position >= len(offsets)
                    or end_position >= len(offsets)
                ):
                    continue

                char_start = offsets[
                    start_position
                ][0]

                char_end = offsets[
                    end_position
                ][1]

                if char_end <= char_start:
                    continue

                answer = context[
                    char_start:char_end
                ].strip()

                if not answer:
                    continue

                candidates.append(
                    {
                        "answer": answer,
                        "score":
                            start_score
                            + end_score,
                        "start_char":
                            char_start,
                        "end_char":
                            char_end,
                        "feature_index":
                            feature_index,
                    }
                )

    # ---------------------------------------------------------------
    # 9. Select best candidate
    # ---------------------------------------------------------------

    if not candidates:

        elapsed = (
            time.time()
            - start_time
        )

        return {
            "answer": "",
            "score": float("-inf"),
            "start_char": None,
            "end_char": None,
            "feature_index": None,
            "num_features":
                len(features),
            "num_candidates": 0,
            "inference_time":
                elapsed,
        }

    candidates.sort(
        key=lambda x: x["score"],
        reverse=True,
    )

    best = candidates[0]

    elapsed = (
        time.time()
        - start_time
    )

    return {
        "answer":
            best["answer"],

        "score":
            best["score"],

        "start_char":
            best["start_char"],

        "end_char":
            best["end_char"],

        "feature_index":
            best["feature_index"],

        "num_features":
            len(features),

        "num_candidates":
            len(candidates),

        "inference_time":
            elapsed,
    }


# =====================================================================
# Locate the already-loaded SQuAD validation split
# =====================================================================

validation_source = None

for variable_name in [
    "raw_datasets",
    "squad",
    "squad_data",
]:

    if variable_name in globals():

        candidate = globals()[
            variable_name
        ]

        if "validation" in candidate:

            validation_source = candidate[
                "validation"
            ]

            break


if validation_source is None:

    raise RuntimeError(
        "SQuAD validation split was not found "
        "in the current runtime."
    )


# =====================================================================
# Real SQuAD example
# =====================================================================

real_example = validation_source[0]

real_question = (
    real_example["question"]
)

real_context = (
    real_example["context"]
)

real_reference = (
    real_example["answers"]["text"][0]
)

real_result = answer_question(
    question=real_question,
    context=real_context,
)


# =====================================================================
# Output
# =====================================================================

print("=" * 70)
print("CLEAN PRODUCTION INFERENCE TEST")
print("=" * 70)

print("\n[1] Real SQuAD example")

print(
    "Question       :",
    real_question
)

print(
    "Reference      :",
    real_reference
)

print(
    "Prediction      :",
    real_result["answer"]
)

print(
    "Score           :",
    f"{real_result['score']:.4f}"
)

print(
    "Features        :",
    real_result["num_features"]
)

print(
    "Best feature    :",
    real_result["feature_index"]
)

print(
    "Candidates      :",
    real_result["num_candidates"]
)

print(
    "Inference time  :",
    f"{real_result['inference_time']:.4f} sec"
)

print("\n[2] Validation")

if real_result["answer"].strip():

    print(
        "PASS — Real SQuAD inference "
        "returned a non-empty answer."
    )

else:

    print(
        "FAIL — Real SQuAD inference "
        "returned an empty answer."
    )

print("\n" + "=" * 70)
print("CLEAN PRODUCTION INFERENCE COMPLETE")
print("=" * 70)

CLEAN PRODUCTION INFERENCE TEST

[1] Real SQuAD example
Question       : Which NFL team represented the AFC at Super Bowl 50?
Reference      : Denver Broncos
Prediction      : Denver Broncos
Score           : 15.7734
Features        : 1
Best feature    : 0
Candidates      : 214
Inference time  : 0.0304 sec

[2] Validation
PASS — Real SQuAD inference returned a non-empty answer.

CLEAN PRODUCTION INFERENCE COMPLETE


In [41]:
# Task 6 — Question Answering with Transformers
# Step 25: Package Final Model for Local Machine

import os
import shutil

FINAL_MODEL_DIR = "/content/qa_distilbert_squad/final_model"
MODEL_ZIP_BASE = "/content/distilbert-squad-final-model"

print("=" * 70)
print("PACKAGING FINAL MODEL")
print("=" * 70)

# ---------------------------------------------------------------------
# 1. Verify final model directory
# ---------------------------------------------------------------------

if not os.path.isdir(FINAL_MODEL_DIR):
    raise FileNotFoundError(
        f"Final model directory not found: {FINAL_MODEL_DIR}"
    )

print("\n[1] Final model directory")
print(FINAL_MODEL_DIR)

# ---------------------------------------------------------------------
# 2. Show files
# ---------------------------------------------------------------------

print("\n[2] Files")

total_size = 0

for root, dirs, files in os.walk(FINAL_MODEL_DIR):

    for filename in sorted(files):

        path = os.path.join(
            root,
            filename
        )

        size = os.path.getsize(path)

        total_size += size

        relative_path = os.path.relpath(
            path,
            FINAL_MODEL_DIR
        )

        print(
            f"{relative_path:<35} "
            f"{size / (1024 ** 2):>8.2f} MB"
        )

print(
    f"\nTotal size: "
    f"{total_size / (1024 ** 2):.2f} MB"
)

# ---------------------------------------------------------------------
# 3. Create ZIP
# ---------------------------------------------------------------------

zip_path = shutil.make_archive(
    MODEL_ZIP_BASE,
    "zip",
    root_dir=FINAL_MODEL_DIR
)

zip_size = os.path.getsize(zip_path)

print("\n[3] ZIP created")

print(
    "Path:",
    zip_path
)

print(
    "ZIP size:",
    f"{zip_size / (1024 ** 2):.2f} MB"
)

# ---------------------------------------------------------------------
# 4. Final validation
# ---------------------------------------------------------------------

if os.path.isfile(zip_path):

    print(
        "\nPASS — Final model packaged successfully."
    )

else:

    raise RuntimeError(
        "Model ZIP was not created."
    )

print("\n" + "=" * 70)
print("MODEL PACKAGING COMPLETE")
print("=" * 70)

PACKAGING FINAL MODEL

[1] Final model directory
/content/qa_distilbert_squad/final_model

[2] Files
config.json                             0.00 MB
model.safetensors                     253.17 MB
tokenizer.json                          0.68 MB
tokenizer_config.json                   0.00 MB
training_args.bin                       0.00 MB

Total size: 253.86 MB

[3] ZIP created
Path: /content/distilbert-squad-final-model.zip
ZIP size: 233.66 MB

PASS — Final model packaged successfully.

MODEL PACKAGING COMPLETE


In [42]:
from google.colab import files

MODEL_ZIP_PATH = "/content/distilbert-squad-final-model.zip"

files.download(MODEL_ZIP_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>